In [1]:
import json
import math
import random
from collections import defaultdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import pairwise_distances

pd.set_option("display.max_columns", None)

# Paths, config

In [2]:
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)

project_dir = Path(".")
step1_dir = project_dir / "results_step1"
step2_dir = project_dir / "results_step2"
step3_dir = project_dir / "results_step3"
step4_dir = project_dir / "results_step4_v2"

step5_6_dir = project_dir / "results_step5_6"
step5_6_dir.mkdir(parents=True, exist_ok=True)

group_size = 3
groups_per_type = 50

top_k = 10

hybrid_rerank_size = 300

lambda_grid = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]
alpha_grid = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]

assert group_size == 3, "this notebook version is configured for group_size = 3"

print("group_size:", group_size)
print("groups_per_type:", groups_per_type)
print("top_k:", top_k)
print("hybrid_rerank_size:", hybrid_rerank_size)
print("lambda_grid:", lambda_grid)
print("alpha_grid:", alpha_grid)

group_size: 3
groups_per_type: 50
top_k: 10
hybrid_rerank_size: 300
lambda_grid: [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]
alpha_grid: [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]


# 1 Load data and previous step artifacts

In [3]:
train_split = pd.read_parquet(step1_dir / "train_split_model_universe.parquet")
validation_split = pd.read_parquet(step1_dir / "validation_split_model_universe.parquet")
test_split = pd.read_parquet(step1_dir / "test_split_model_universe.parquet")
movies = pd.read_parquet(step1_dir / "movies.parquet")

user_mapping = pd.read_parquet(step2_dir / "user_index_mapping.parquet")
item_mapping = pd.read_parquet(step2_dir / "item_index_mapping.parquet")

user_embeddings = np.load(step2_dir / "user_embeddings_p_u.npy").astype(np.float32)
item_embeddings = np.load(step2_dir / "item_embeddings_q_i.npy").astype(np.float32)
item_bias = np.load(step2_dir / "item_bias.npy").astype(np.float32)

user_clusters = pd.read_csv(step3_dir / "user_dbscan_labels.csv")

bayes_artifact = joblib.load(step4_dir / "best_bayesian_categorical_nb_v2.joblib")

with open(step4_dir / "best_knn_config.json", "r") as file:
    knn_config = json.load(file)

knn_score_matrix_path = step4_dir / "best_rating_user_knn_score_matrix.npy"
if not knn_score_matrix_path.exists():
    raise FileNotFoundError(
        "Missing independent rating-user-kNN score matrix. "
        "Run the updated Step 4 notebook before Step 5-6."
    )

step4_pairwise_metrics = pd.read_csv(step4_dir / "compact_model_comparison.csv")

print("train split:", train_split.shape)
print("validation split:", validation_split.shape)
print("test split:", test_split.shape)
print("movies:", movies.shape)
print("user embeddings:", user_embeddings.shape)
print("item embeddings:", item_embeddings.shape)
print("user clusters:", user_clusters.shape)
print("kNN config:", knn_config)
display(step4_pairwise_metrics)

train split: (759306, 5)
validation split: (100787, 5)
test split: (100564, 5)
movies: (3883, 3)
user embeddings: (4722, 64)
item embeddings: (3250, 64)
user clusters: (4722, 13)
kNN config: {'k': 50, 'weighting': 'uniform', 'metric': 'cosine', 'neighbor_space': 'centered_train_rating_vectors', 'score_definition': "user mean plus weighted average of neighbors' centered train ratings, with item-mean fallback", 'uses_mf_scores': False, 'uses_mf_embeddings': False, 'uses_dbscan': False}


,model,model_family,split,feature_set,accuracy,macro_user_accuracy,auc,brier_score,log_loss,mean_forward_probability,n_bins,strategy,alpha,k,weighting
0,mf_baseline,matrix_factorization,test,mf_margin,0.817919,0.774139,0.903074,0.126110,0.394358,0.753358,NaN,NaN,NaN,NaN,NaN
1,bayes_user_item_interactions,bayesian_categorical_nb,test,user_item_interactions,0.811605,0.765294,0.897973,0.170505,1.668127,0.809655,16.0,quantile,0.1,NaN,NaN
2,rating_user_knn_k_50_uniform,rating_user_knn,test,centered_train_ratings,0.778725,0.713992,0.859784,0.159424,0.490201,0.646041,NaN,NaN,NaN,50.0,uniform
3,mf_baseline,matrix_factorization,train_sample,mf_margin,0.939077,0.947161,0.985116,0.050974,0.185189,0.861701,NaN,NaN,NaN,NaN,NaN
4,rating_user_knn_k_50_uniform,rating_user_knn,train_sample,centered_train_ratings,0.912617,0.932404,0.969226,0.098573,0.349697,0.724308,NaN,NaN,NaN,50.0,uniform
5,bayes_user_item_interactions,bayesian_categorical_nb,train_sample,user_item_interactions,0.919457,0.927988,0.973339,0.071218,0.579676,0.916964,16.0,quantile,0.1,NaN,NaN
6,mf_baseline,matrix_factorization,validation,mf_margin,0.808433,0.757608,0.896024,0.131186,0.404075,0.741683,NaN,NaN,NaN,NaN,NaN
7,bayes_user_item_interactions,bayesian_categorical_nb,validation,user_item_interactions,0.800453,0.748360,0.889484,0.179916,1.658191,0.798488,16.0,quantile,0.1,NaN,NaN
8,rating_user_knn_k_50_uniform,rating_user_knn,validation,centered_train_ratings,0.773410,0.706558,0.855531,0.163029,0.498905,0.637938,NaN,NaN,NaN,50.0,uniform


### 1.1 Mappings

In [4]:
user_id_to_idx = dict(zip(user_mapping["user_id"], user_mapping["user_idx"]))
user_idx_to_id = dict(zip(user_mapping["user_idx"], user_mapping["user_id"]))

movie_id_to_item_idx = dict(zip(item_mapping["movie_id"], item_mapping["item_idx"]))
item_idx_to_movie_id = dict(zip(item_mapping["item_idx"], item_mapping["movie_id"]))

if "title" in movies.columns:
    movie_id_to_title = dict(zip(movies["movie_id"], movies["title"]))
else:
    movie_id_to_title = {movie_id: str(movie_id) for movie_id in item_mapping["movie_id"]}

n_users = user_embeddings.shape[0]
n_items = item_embeddings.shape[0]
all_item_idx = np.arange(n_items, dtype=np.int32)

assert len(user_mapping) == n_users
assert len(item_mapping) == n_items
assert len(item_bias) == n_items
assert user_embeddings.shape[1] == item_embeddings.shape[1]

print("users:", n_users)
print("items:", n_items)
print("embedding_dim:", user_embeddings.shape[1])

users: 4722
items: 3250
embedding_dim: 64


## 1.2 Indexed splits, seen sets, and relevance sets

In [5]:
def add_indices_to_ratings(ratings_df):
    ratings_df = ratings_df.copy()

    ratings_df["user_idx"] = ratings_df["user_id"].map(user_id_to_idx)
    ratings_df["item_idx"] = ratings_df["movie_id"].map(movie_id_to_item_idx)

    ratings_df = ratings_df.dropna(subset=["user_idx", "item_idx"]).copy()
    ratings_df["user_idx"] = ratings_df["user_idx"].astype(np.int32)
    ratings_df["item_idx"] = ratings_df["item_idx"].astype(np.int32)

    return ratings_df


def build_seen_dict(ratings_df):
    return (
        ratings_df
        .groupby("user_idx")["item_idx"]
        .apply(lambda values: set(values.astype(int)))
        .to_dict()
    )


def build_relevance_dict(ratings_df, min_rating=4):
    return (
        ratings_df
        .loc[ratings_df["rating"] >= min_rating]
        .groupby("user_idx")["item_idx"]
        .apply(lambda values: set(values.astype(int)))
        .to_dict()
    )


train_split_idx = add_indices_to_ratings(train_split)
validation_split_idx = add_indices_to_ratings(validation_split)
test_split_idx = add_indices_to_ratings(test_split)

train_seen_by_user_idx = build_seen_dict(train_split_idx)
validation_seen_by_user_idx = build_seen_dict(validation_split_idx)

def union_user_sets(first, second):
    keys = set(first.keys()).union(second.keys())
    return {
        int(key): set(first.get(key, set())).union(second.get(key, set()))
        for key in keys
    }

train_validation_seen_by_user_idx = union_user_sets(
    train_seen_by_user_idx,
    validation_seen_by_user_idx,
)

validation_relevant_by_user_idx = build_relevance_dict(validation_split_idx, min_rating=4)
test_relevant_by_user_idx = build_relevance_dict(test_split_idx, min_rating=4)

print("train users with seen items:", len(train_seen_by_user_idx))
print("validation users with seen items:", len(validation_seen_by_user_idx))
print("train+validation users with seen items:", len(train_validation_seen_by_user_idx))
print("validation users with relevant items:", len(validation_relevant_by_user_idx))
print("test users with relevant items:", len(test_relevant_by_user_idx))

train users with seen items: 4722
validation users with seen items: 4722
train+validation users with seen items: 4722
validation users with relevant items: 4641
test users with relevant items: 4623


# 2 Evaluation helpers used for validation tuning and test evaluation

In [6]:
def dcg_from_gains(gains):
    gains = np.asarray(gains, dtype=np.float64)

    if len(gains) == 0:
        return 0.0

    discounts = 1.0 / np.log2(np.arange(2, len(gains) + 2))
    return float(np.sum(gains * discounts))


def ndcg_from_gains(gains, ideal_gains):
    dcg_value = dcg_from_gains(gains)
    ideal_dcg_value = dcg_from_gains(sorted(ideal_gains, reverse=True)[:len(gains)])

    if ideal_dcg_value <= 0:
        return 0.0

    return float(dcg_value / ideal_dcg_value)


def precision_at_k(recommended_items, relevant_items, k):
    recommended_items = list(recommended_items)[:k]
    relevant_items = set(relevant_items)

    if k <= 0:
        return 0.0

    hits = sum(1 for item_idx in recommended_items if int(item_idx) in relevant_items)
    return float(hits / k)


def recall_at_k(recommended_items, relevant_items, k):
    recommended_items = list(recommended_items)[:k]
    relevant_items = set(relevant_items)

    if len(relevant_items) == 0:
        return 0.0

    hits = sum(1 for item_idx in recommended_items if int(item_idx) in relevant_items)
    return float(hits / len(relevant_items))


def hit_rate_at_k(recommended_items, relevant_items, k):
    recommended_items = list(recommended_items)[:k]
    relevant_items = set(relevant_items)
    return float(any(int(item_idx) in relevant_items for item_idx in recommended_items))


def ndcg_at_k_binary(recommended_items, relevant_items, k):
    recommended_items = list(recommended_items)[:k]
    relevant_items = set(relevant_items)

    gains = [1.0 if int(item_idx) in relevant_items else 0.0 for item_idx in recommended_items]
    ideal_gains = [1.0] * min(k, len(relevant_items))

    return ndcg_from_gains(gains, ideal_gains)


def jain_index(values):
    values = np.asarray(values, dtype=np.float64)
    denominator = len(values) * np.sum(values ** 2)

    if denominator <= 0:
        return 0.0

    return float((np.sum(values) ** 2) / denominator)


def get_group_gain_dict(user_idx_list, relevant_by_user_idx, eligible_items=None):
    if eligible_items is None:
        eligible_set = None
    else:
        eligible_set = set(int(item_idx) for item_idx in eligible_items)

    gain_dict = defaultdict(float)

    for user_idx in user_idx_list:
        relevant_items = relevant_by_user_idx.get(int(user_idx), set())

        for item_idx in relevant_items:
            item_idx = int(item_idx)

            if eligible_set is not None and item_idx not in eligible_set:
                continue

            gain_dict[item_idx] += 1.0 / len(user_idx_list)

    return dict(gain_dict)


def intra_list_embedding_distance(item_idx_list):
    item_idx_list = list(item_idx_list)

    if len(item_idx_list) < 2:
        return 0.0

    vectors = item_embeddings[np.asarray(item_idx_list, dtype=np.int32)]
    distances = pairwise_distances(vectors, metric="cosine")
    upper_values = distances[np.triu_indices_from(distances, k=1)]

    return float(np.mean(upper_values))

# 3 Disjoint DBSCAN based groups

In [7]:
def normalized_entropy(values):
    values = list(values)

    if len(values) == 0:
        return 0.0

    counts = pd.Series(values).value_counts(normalize=True).values

    if len(counts) <= 1:
        return 0.0

    entropy = -float(np.sum(counts * np.log(counts)))
    return entropy / math.log(len(counts))


def average_pairwise_cosine_distance(user_idx_list):
    vectors = user_embeddings[np.asarray(user_idx_list, dtype=np.int32)]

    if len(vectors) < 2:
        return 0.0

    distances = pairwise_distances(vectors, metric="cosine")
    upper_values = distances[np.triu_indices_from(distances, k=1)]

    return float(np.mean(upper_values))


def choose_disjoint(pool_values, needed_count, rng, pool_name, used_users):
    pool_values = [int(value) for value in pool_values if int(value) not in used_users]

    if len(pool_values) < needed_count:
        raise ValueError(
            f"not enough unused users in {pool_name}: needed {needed_count}, available {len(pool_values)}"
        )

    selected = rng.choice(np.asarray(pool_values, dtype=np.int32), size=needed_count, replace=False).tolist()
    used_users.update(int(value) for value in selected)

    return selected


def make_group_record(group_id, group_type, user_idx_list):
    cluster_values = [
        int(user_idx_to_cluster[int(user_idx)])
        for user_idx in user_idx_list
    ]

    cluster_counts = pd.Series(cluster_values).value_counts()

    majority_cluster = None
    minority_cluster = None

    if len(cluster_counts) > 1 and cluster_counts.min() < cluster_counts.max():
        majority_cluster = int(cluster_counts.idxmax())
        minority_cluster = int(cluster_counts.idxmin())

    return {
        "group_id": group_id,
        "group_type": group_type,
        "user_idx_list": [int(value) for value in user_idx_list],
        "user_id_list": [int(user_idx_to_id[int(value)]) for value in user_idx_list],
        "cluster_list": cluster_values,
        "cluster_entropy": normalized_entropy(cluster_values),
        "avg_pairwise_embedding_distance": average_pairwise_cosine_distance(user_idx_list),
        "majority_cluster": majority_cluster,
        "minority_cluster": minority_cluster,
        "noise_share": float(np.mean(np.asarray(cluster_values) == -1)),
    }

In [8]:
# Build two separate group sets:
# 1. validation_groups: used only for aggregation/fairness hyperparameter tuning
# 2. test_groups: used only for final recommendation generation and test evaluation
#
# This avoids selecting validation-tuning groups using test-period relevance information.

user_idx_to_cluster = dict(
    zip(
        user_clusters["user_idx"].astype(int),
        user_clusters["user_cluster"].astype(int),
    )
)


def get_cluster_pools_for_eligible_users(eligible_user_idx, split_name):
    eligible_user_idx = set(int(value) for value in eligible_user_idx)

    eligible_user_clusters = user_clusters[
        user_clusters["user_idx"].astype(int).isin(eligible_user_idx)
    ].copy()

    cluster_0_users = eligible_user_clusters.loc[
        eligible_user_clusters["user_cluster"] == 0,
        "user_idx",
    ].astype(int).values

    cluster_1_users = eligible_user_clusters.loc[
        eligible_user_clusters["user_cluster"] == 1,
        "user_idx",
    ].astype(int).values

    noise_users = eligible_user_clusters.loc[
        eligible_user_clusters["user_cluster"] == -1,
        "user_idx",
    ].astype(int).values

    print(f"{split_name} eligible cluster 0 users:", len(cluster_0_users))
    print(f"{split_name} eligible cluster 1 users:", len(cluster_1_users))
    print(f"{split_name} eligible noise users:", len(noise_users))

    return {
        "eligible_user_clusters": eligible_user_clusters,
        "cluster_0_users": cluster_0_users,
        "cluster_1_users": cluster_1_users,
        "noise_users": noise_users,
    }


def build_disjoint_dbscan_groups(eligible_user_idx, split_name, seed_offset):
    pools = get_cluster_pools_for_eligible_users(
        eligible_user_idx=eligible_user_idx,
        split_name=split_name,
    )

    cluster_0_users = pools["cluster_0_users"]
    cluster_1_users = pools["cluster_1_users"]
    noise_users = pools["noise_users"]

    rng = np.random.default_rng(random_seed + seed_offset)
    used_users = set()
    output_groups = []
    group_counter = 0

    group_id_prefix = f"{split_name}_group"

    # Homogeneous cluster 0 groups
    chosen_c0 = choose_disjoint(
        pool_values=cluster_0_users,
        needed_count=groups_per_type * group_size,
        rng=rng,
        pool_name=f"{split_name} cluster_0_users",
        used_users=used_users,
    )

    for start in range(0, len(chosen_c0), group_size):
        group_counter += 1
        output_groups.append(
            make_group_record(
                f"{group_id_prefix}_{group_counter:04d}",
                "homogeneous_cluster_0",
                chosen_c0[start:start + group_size],
            )
        )

    # Homogeneous cluster 1 groups
    chosen_c1 = choose_disjoint(
        pool_values=cluster_1_users,
        needed_count=groups_per_type * group_size,
        rng=rng,
        pool_name=f"{split_name} cluster_1_users",
        used_users=used_users,
    )

    for start in range(0, len(chosen_c1), group_size):
        group_counter += 1
        output_groups.append(
            make_group_record(
                f"{group_id_prefix}_{group_counter:04d}",
                "homogeneous_cluster_1",
                chosen_c1[start:start + group_size],
            )
        )

    # Diverse balanced groups: 1 cluster 0 + 1 cluster 1 + 1 noise
    balanced_c0 = choose_disjoint(
        cluster_0_users,
        groups_per_type,
        rng,
        f"{split_name} balanced cluster 0",
        used_users,
    )

    balanced_c1 = choose_disjoint(
        cluster_1_users,
        groups_per_type,
        rng,
        f"{split_name} balanced cluster 1",
        used_users,
    )

    balanced_noise = choose_disjoint(
        noise_users,
        groups_per_type,
        rng,
        f"{split_name} balanced noise",
        used_users,
    )

    for group_pos in range(groups_per_type):
        group_counter += 1
        user_idx_list = [
            balanced_c0[group_pos],
            balanced_c1[group_pos],
            balanced_noise[group_pos],
        ]
        rng.shuffle(user_idx_list)

        output_groups.append(
            make_group_record(
                f"{group_id_prefix}_{group_counter:04d}",
                "diverse_balanced",
                user_idx_list,
            )
        )

    # Diverse minority groups:
    # half are 2x cluster 0 + 1x cluster 1,
    # half are 2x cluster 1 + 1x cluster 0.
    minority_first_count = groups_per_type // 2
    minority_second_count = groups_per_type - minority_first_count

    minority_c0_needed = minority_first_count * 2 + minority_second_count * 1
    minority_c1_needed = minority_first_count * 1 + minority_second_count * 2

    minority_c0 = choose_disjoint(
        cluster_0_users,
        minority_c0_needed,
        rng,
        f"{split_name} minority cluster 0",
        used_users,
    )

    minority_c1 = choose_disjoint(
        cluster_1_users,
        minority_c1_needed,
        rng,
        f"{split_name} minority cluster 1",
        used_users,
    )

    c0_cursor = 0
    c1_cursor = 0

    for _ in range(minority_first_count):
        group_counter += 1
        user_idx_list = (
            minority_c0[c0_cursor:c0_cursor + 2]
            + minority_c1[c1_cursor:c1_cursor + 1]
        )
        c0_cursor += 2
        c1_cursor += 1
        rng.shuffle(user_idx_list)

        output_groups.append(
            make_group_record(
                f"{group_id_prefix}_{group_counter:04d}",
                "diverse_minority",
                user_idx_list,
            )
        )

    for _ in range(minority_second_count):
        group_counter += 1
        user_idx_list = (
            minority_c1[c1_cursor:c1_cursor + 2]
            + minority_c0[c0_cursor:c0_cursor + 1]
        )
        c1_cursor += 2
        c0_cursor += 1
        rng.shuffle(user_idx_list)

        output_groups.append(
            make_group_record(
                f"{group_id_prefix}_{group_counter:04d}",
                "diverse_minority",
                user_idx_list,
            )
        )

    expected_group_count = 4 * groups_per_type
    used_user_count = sum(len(group["user_idx_list"]) for group in output_groups)
    unique_user_count = len({user_idx for group in output_groups for user_idx in group["user_idx_list"]})

    assert len(output_groups) == expected_group_count
    assert used_user_count == expected_group_count * group_size
    assert unique_user_count == used_user_count, f"{split_name} groups are not disjoint"

    print(f"{split_name} groups:", len(output_groups))
    print(f"{split_name} unique users used:", unique_user_count)
    print(f"{split_name} group-member slots:", used_user_count)

    return output_groups, pools["eligible_user_clusters"]


validation_groups, validation_eligible_user_clusters = build_disjoint_dbscan_groups(
    eligible_user_idx=validation_relevant_by_user_idx.keys(),
    split_name="validation",
    seed_offset=101,
)

test_groups, test_eligible_user_clusters = build_disjoint_dbscan_groups(
    eligible_user_idx=test_relevant_by_user_idx.keys(),
    split_name="test",
    seed_offset=202,
)

# Keep the old variable name for downstream final recommendation/evaluation cells.
# From this point onward, `groups` means final test groups.
groups = test_groups

print("validation groups:", len(validation_groups))
print("test groups:", len(test_groups))

validation eligible cluster 0 users: 1258
validation eligible cluster 1 users: 1502
validation eligible noise users: 1881
validation groups: 200
validation unique users used: 600
validation group-member slots: 600
test eligible cluster 0 users: 1252
test eligible cluster 1 users: 1500
test eligible noise users: 1871
test groups: 200
test unique users used: 600
test group-member slots: 600
validation groups: 200
test groups: 200


In [9]:
def make_group_dataframes(groups_to_summarize):
    group_summary = pd.DataFrame([
        {
            "group_id": group["group_id"],
            "group_type": group["group_type"],
            "user_idx_list": json.dumps(group["user_idx_list"]),
            "user_id_list": json.dumps(group["user_id_list"]),
            "cluster_list": json.dumps(group["cluster_list"]),
            "cluster_entropy": group["cluster_entropy"],
            "avg_pairwise_embedding_distance": group["avg_pairwise_embedding_distance"],
            "majority_cluster": group["majority_cluster"],
            "minority_cluster": group["minority_cluster"],
            "noise_share": group["noise_share"],
        }
        for group in groups_to_summarize
    ])

    group_member_rows = []

    for group in groups_to_summarize:
        for member_position, user_idx in enumerate(group["user_idx_list"]):
            group_member_rows.append(
                {
                    "group_id": group["group_id"],
                    "group_type": group["group_type"],
                    "member_position": member_position,
                    "user_idx": int(user_idx),
                    "user_id": int(user_idx_to_id[int(user_idx)]),
                    "user_cluster": int(user_idx_to_cluster[int(user_idx)]),
                }
            )

    group_members = pd.DataFrame(group_member_rows)

    return group_summary, group_members


def validate_group_dataframes(group_summary, group_members, split_name):
    expected_group_count = 4 * groups_per_type

    assert len(group_summary) == expected_group_count
    assert group_summary["group_type"].value_counts().eq(groups_per_type).all()
    assert group_members.groupby("group_id").size().eq(group_size).all()
    assert group_members["user_idx"].nunique() == len(group_members), f"{split_name} groups are not disjoint"

    homogeneous_groups = group_members[
        group_members["group_type"].isin(["homogeneous_cluster_0", "homogeneous_cluster_1"])
    ]
    assert homogeneous_groups.groupby("group_id")["user_cluster"].nunique().eq(1).all()

    balanced_counts = (
        group_members[group_members["group_type"] == "diverse_balanced"]
        .groupby(["group_id", "user_cluster"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=[-1, 0, 1], fill_value=0)
    )
    assert (balanced_counts[0] == 1).all()
    assert (balanced_counts[1] == 1).all()
    assert (balanced_counts[-1] == 1).all()

    minority_counts = (
        group_members[group_members["group_type"] == "diverse_minority"]
        .groupby(["group_id", "user_cluster"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=[-1, 0, 1], fill_value=0)
    )

    valid_minority_pattern = (
        ((minority_counts[0] == 2) & (minority_counts[1] == 1) & (minority_counts[-1] == 0))
        |
        ((minority_counts[0] == 1) & (minority_counts[1] == 2) & (minority_counts[-1] == 0))
    )
    assert valid_minority_pattern.all()

    print(f"{split_name} group dataframe checks passed")


validation_group_summary_df, validation_group_members_df = make_group_dataframes(validation_groups)
group_summary_df, group_members_df = make_group_dataframes(test_groups)

validate_group_dataframes(validation_group_summary_df, validation_group_members_df, "validation")
validate_group_dataframes(group_summary_df, group_members_df, "test")

display(group_summary_df["group_type"].value_counts())
display(group_members_df.groupby(["group_type", "user_cluster"]).size().reset_index(name="user_slots"))
display(group_summary_df.head())

validation group dataframe checks passed
test group dataframe checks passed


group_type
homogeneous_cluster_0    50
homogeneous_cluster_1    50
diverse_balanced         50
diverse_minority         50
Name: count, dtype: int64

,group_type,user_cluster,user_slots
0,diverse_balanced,-1,50
1,diverse_balanced,0,50
2,diverse_balanced,1,50
3,diverse_minority,0,75
4,diverse_minority,1,75
5,homogeneous_cluster_0,0,150
6,homogeneous_cluster_1,1,150


,group_id,group_type,user_idx_list,user_id_list,cluster_list,cluster_entropy,avg_pairwise_embedding_distance,majority_cluster,minority_cluster,noise_share
0,test_group_0001,homogeneous_cluster_0,"[4411, 2090, 2795]","[5648, 2698, 3599]","[0, 0, 0]",0.0,1.055191,NaN,NaN,0.0
1,test_group_0002,homogeneous_cluster_0,"[2106, 2698, 2566]","[2730, 3476, 3306]","[0, 0, 0]",0.0,0.795931,NaN,NaN,0.0
2,test_group_0003,homogeneous_cluster_0,"[1824, 163, 729]","[2329, 212, 935]","[0, 0, 0]",0.0,0.808395,NaN,NaN,0.0
3,test_group_0004,homogeneous_cluster_0,"[3185, 1873, 3094]","[4063, 2400, 3955]","[0, 0, 0]",0.0,0.587802,NaN,NaN,0.0
4,test_group_0005,homogeneous_cluster_0,"[3136, 2909, 1230]","[4003, 3734, 1575]","[0, 0, 0]",0.0,1.009713,NaN,NaN,0.0


In [10]:
# Save final test groups using the original filenames for backward compatibility.
group_summary_df.to_csv(step5_6_dir / "step5_groups_summary.csv", index=False)
group_members_df.to_csv(step5_6_dir / "step5_group_members.csv", index=False)

group_summary_df.to_parquet(step5_6_dir / "step5_groups_summary.parquet", index=False)
group_members_df.to_parquet(step5_6_dir / "step5_group_members.parquet", index=False)

# Also save explicit split-specific group files.
validation_group_summary_df.to_csv(step5_6_dir / "step5_validation_groups_summary.csv", index=False)
validation_group_members_df.to_csv(step5_6_dir / "step5_validation_group_members.csv", index=False)

group_summary_df.to_csv(step5_6_dir / "step5_test_groups_summary.csv", index=False)
group_members_df.to_csv(step5_6_dir / "step5_test_group_members.csv", index=False)

validation_group_summary_df.to_parquet(step5_6_dir / "step5_validation_groups_summary.parquet", index=False)
validation_group_members_df.to_parquet(step5_6_dir / "step5_validation_group_members.parquet", index=False)

group_summary_df.to_parquet(step5_6_dir / "step5_test_groups_summary.parquet", index=False)
group_members_df.to_parquet(step5_6_dir / "step5_test_group_members.parquet", index=False)

print("saved validation and final test group construction outputs")

saved validation and final test group construction outputs


# 4 Score matrices and aggregation rules

In [11]:
mf_score_matrix = user_embeddings @ item_embeddings.T
mf_score_matrix = mf_score_matrix + item_bias.reshape(1, -1)
mf_score_matrix = mf_score_matrix.astype(np.float32)

knn_score_matrix = np.load(knn_score_matrix_path).astype(np.float32)

assert mf_score_matrix.shape == (n_users, n_items)
assert knn_score_matrix.shape == (n_users, n_items)
assert not bool(knn_config.get("uses_mf_scores", True)), "Step 5 expects the independent rating-user-kNN from updated Step 4"

print("mf score matrix:", mf_score_matrix.shape)
print("independent rating-user-kNN score matrix:", knn_score_matrix.shape)

mf score matrix: (4722, 3250)
independent rating-user-kNN score matrix: (4722, 3250)


In [12]:
def normalize_member_scores(member_scores, eps=1e-8):
    """
    Normalize scores separately for each group member.

    Input shape:
        member_scores: [n_members, n_candidate_items]

    Why per-member?
        Fairness-aware aggregation should compare each member's relative preferences.
        Global min-max normalization can let one member dominate if their score range
        is wider than others.
    """
    member_scores = np.asarray(member_scores, dtype=np.float32)

    score_min = np.min(member_scores, axis=1, keepdims=True)
    score_max = np.max(member_scores, axis=1, keepdims=True)
    denom = score_max - score_min

    normalized = np.divide(
        member_scores - score_min,
        denom + eps,
        out=np.zeros_like(member_scores, dtype=np.float32),
        where=denom > eps,
    )

    return normalized.astype(np.float32)


def normalize_vector(values, eps=1e-8):
    """
    Normalize a single candidate-score vector to [0, 1].

    Used inside dbscan_minority_nash so that Nash, minority satisfaction,
    and worst-member components are on comparable scales before mixing.
    """
    values = np.asarray(values, dtype=np.float32)

    value_min = float(np.min(values))
    value_max = float(np.max(values))

    if value_max - value_min <= eps:
        return np.zeros_like(values, dtype=np.float32)

    return ((values - value_min) / (value_max - value_min + eps)).astype(np.float32)


def get_minority_positions(member_clusters, minority_cluster):
    if minority_cluster is None or pd.isna(minority_cluster):
        return []

    return [
        pos
        for pos, cluster_value in enumerate(member_clusters)
        if int(cluster_value) == int(minority_cluster)
    ]


def aggregate_member_scores(
    member_scores,
    aggregation,
    member_clusters=None,
    minority_cluster=None,
    lambda_std=0.0,
    alpha_minority=0.0,
):
    scores = normalize_member_scores(member_scores)

    if aggregation == "average":
        return scores.mean(axis=0)

    if aggregation == "least_misery":
        return scores.min(axis=0)

    if aggregation == "tuned_fairness":
        # Utility minus disagreement penalty.
        # Higher lambda_std means stronger penalty for items that split the group.
        return scores.mean(axis=0) - lambda_std * scores.std(axis=0)

    if aggregation == "nash_welfare":
        # Raw Nash score is fine here because it is used alone for ranking.
        return np.log(scores + 1e-6).sum(axis=0)

    if aggregation == "dbscan_minority_nash":
        minority_positions = get_minority_positions(member_clusters, minority_cluster)

        nash_score = np.log(scores + 1e-6).sum(axis=0)
        nash_score = normalize_vector(nash_score)

        if len(minority_positions) == 0:
            return nash_score

        minority_positions = np.asarray(minority_positions, dtype=np.int32)

        minority_score = scores[minority_positions].mean(axis=0)
        minority_score = normalize_vector(minority_score)

        worst_member_score = scores.min(axis=0)
        worst_member_score = normalize_vector(worst_member_score)

        # Main term: Nash group welfare.
        # Minority term: explicit DBSCAN minority-cluster protection.
        # Worst-member term: prevents the minority boost from selecting items
        # that completely fail another member.
        return (
            nash_score
            + alpha_minority * minority_score
            + 0.25 * alpha_minority * worst_member_score
        )

    raise ValueError(f"unknown aggregation: {aggregation}")


def get_top_items_from_scores(item_idx_array, score_array, top_n):
    item_idx_array = np.asarray(item_idx_array, dtype=np.int32)
    score_array = np.asarray(score_array, dtype=np.float32)

    top_n = min(top_n, len(item_idx_array))

    if top_n <= 0:
        return np.array([], dtype=np.int32), np.array([], dtype=np.float32)

    top_positions = np.argpartition(-score_array, top_n - 1)[:top_n]
    top_positions = top_positions[np.argsort(-score_array[top_positions])]

    return item_idx_array[top_positions], score_array[top_positions]

In [13]:
def get_eligible_items_for_group(user_idx_list, seen_by_user_idx):
    seen_items = set()

    for user_idx in user_idx_list:
        seen_items.update(seen_by_user_idx.get(int(user_idx), set()))

    return np.array(
        [item_idx for item_idx in all_item_idx if int(item_idx) not in seen_items],
        dtype=np.int32,
    )

# 5 Validation tuning for fairness and DBSCAN minority - aware aggregation

In [14]:
def score_group_with_matrix(
    group,
    score_matrix,
    seen_by_user_idx,
    aggregation,
    lambda_std=0.0,
    alpha_minority=0.0,
):
    user_idx_list = group["user_idx_list"]
    eligible_items = get_eligible_items_for_group(user_idx_list, seen_by_user_idx)

    member_scores = score_matrix[np.asarray(user_idx_list, dtype=np.int32)][:, eligible_items]

    group_scores = aggregate_member_scores(
        member_scores=member_scores,
        aggregation=aggregation,
        member_clusters=group["cluster_list"],
        minority_cluster=group["minority_cluster"],
        lambda_std=lambda_std,
        alpha_minority=alpha_minority,
    )

    top_item_idx, top_scores = get_top_items_from_scores(
        item_idx_array=eligible_items,
        score_array=group_scores,
        top_n=top_k,
    )

    return eligible_items, top_item_idx, top_scores


def evaluate_group_recommendation(
    group,
    recommended_items,
    eligible_items,
    relevant_by_user_idx,
):
    user_idx_list = group["user_idx_list"]
    recommended_items = list(recommended_items)[:top_k]
    eligible_item_set = set(int(item_idx) for item_idx in eligible_items)

    group_gain_dict = get_group_gain_dict(
        user_idx_list=user_idx_list,
        relevant_by_user_idx=relevant_by_user_idx,
        eligible_items=eligible_item_set,
    )

    group_gains = [
        group_gain_dict.get(int(item_idx), 0.0)
        for item_idx in recommended_items
    ]

    ideal_group_gains = sorted(group_gain_dict.values(), reverse=True)[:top_k]
    total_group_gain = float(np.sum(list(group_gain_dict.values())))

    group_precision = float(np.sum(group_gains) / top_k)
    group_recall = float(np.sum(group_gains) / total_group_gain) if total_group_gain > 0 else 0.0
    group_ndcg = ndcg_from_gains(group_gains, ideal_group_gains)

    member_satisfactions = []
    member_rows = []

    for user_idx in user_idx_list:
        relevant_items = {
            int(item_idx)
            for item_idx in relevant_by_user_idx.get(int(user_idx), set())
            if int(item_idx) in eligible_item_set
        }

        member_precision = precision_at_k(recommended_items, relevant_items, top_k)
        member_recall = recall_at_k(recommended_items, relevant_items, top_k)
        member_ndcg = ndcg_at_k_binary(recommended_items, relevant_items, top_k)
        member_hit = hit_rate_at_k(recommended_items, relevant_items, top_k)

        member_satisfactions.append(member_ndcg)
        member_rows.append(
            {
                "user_idx": int(user_idx),
                "user_id": int(user_idx_to_id[int(user_idx)]),
                "user_cluster": int(user_idx_to_cluster[int(user_idx)]),
                "member_precision_10": member_precision,
                "member_recall_10": member_recall,
                "member_ndcg_10": member_ndcg,
                "member_hit_10": member_hit,
                "member_satisfaction": member_ndcg,
            }
        )

    member_satisfactions = np.asarray(member_satisfactions, dtype=np.float64)

    cluster_satisfaction_values = {}
    cluster_zero_values = {}

    for cluster_value in sorted(set(group["cluster_list"])):
        cluster_member_satisfactions = [
            satisfaction
            for satisfaction, member_cluster in zip(member_satisfactions, group["cluster_list"])
            if int(member_cluster) == int(cluster_value)
        ]

        cluster_satisfaction_values[int(cluster_value)] = float(np.mean(cluster_member_satisfactions))
        cluster_zero_values[int(cluster_value)] = float(np.mean(np.asarray(cluster_member_satisfactions) == 0.0))

    if len(cluster_satisfaction_values) > 1:
        cluster_satisfaction_gap = float(max(cluster_satisfaction_values.values()) - min(cluster_satisfaction_values.values()))
    else:
        cluster_satisfaction_gap = 0.0

    minority_cluster = group["minority_cluster"]

    if minority_cluster is not None and not pd.isna(minority_cluster):
        minority_cluster_satisfaction = cluster_satisfaction_values.get(int(minority_cluster), np.nan)
        minority_zero_satisfaction_rate = cluster_zero_values.get(int(minority_cluster), np.nan)
    else:
        minority_cluster_satisfaction = np.nan
        minority_zero_satisfaction_rate = np.nan

    metric_row = {
        "group_precision_10": group_precision,
        "group_recall_10": group_recall,
        "group_ndcg_10": group_ndcg,
        "mean_member_ndcg_10": float(np.mean(member_satisfactions)),
        "mean_satisfaction": float(np.mean(member_satisfactions)),
        "std_satisfaction": float(np.std(member_satisfactions)),
        "worst_member_satisfaction": float(np.min(member_satisfactions)),
        "jain_fairness": jain_index(member_satisfactions),
        "minority_cluster_satisfaction": minority_cluster_satisfaction,
        "minority_zero_satisfaction_rate": minority_zero_satisfaction_rate,
        "cluster_satisfaction_gap": cluster_satisfaction_gap,
        "zero_satisfaction_share": float(np.mean(member_satisfactions == 0.0)),
        "member_hit_share": float(np.mean(member_satisfactions > 0.0)),
        "intra_list_embedding_distance": intra_list_embedding_distance(recommended_items),
    }

    return metric_row, member_rows

In [15]:
def evaluate_mf_aggregation_on_validation(aggregation, lambda_std=0.0, alpha_minority=0.0):
    rows = []

    for group in validation_groups:
        eligible_items, top_item_idx, _ = score_group_with_matrix(
            group=group,
            score_matrix=mf_score_matrix,
            seen_by_user_idx=train_seen_by_user_idx,
            aggregation=aggregation,
            lambda_std=lambda_std,
            alpha_minority=alpha_minority,
        )

        metric_row, _ = evaluate_group_recommendation(
            group=group,
            recommended_items=top_item_idx,
            eligible_items=eligible_items,
            relevant_by_user_idx=validation_relevant_by_user_idx,
        )

        rows.append({
            "split": "validation",
            "aggregation": aggregation,
            "lambda_std": float(lambda_std),
            "alpha_minority": float(alpha_minority),
            **metric_row,
        })

    return pd.DataFrame(rows)


validation_average_df = evaluate_mf_aggregation_on_validation("average")
validation_nash_df = evaluate_mf_aggregation_on_validation("nash_welfare")

average_validation_ndcg = float(validation_average_df["group_ndcg_10"].mean())
average_validation_jain = float(validation_average_df["jain_fairness"].mean())
average_validation_zero = float(validation_average_df["zero_satisfaction_share"].mean())
average_validation_worst = float(validation_average_df["worst_member_satisfaction"].mean())

nash_validation_ndcg = float(validation_nash_df["group_ndcg_10"].mean())

print("average validation NDCG@10:", average_validation_ndcg)
print("average validation Jain:", average_validation_jain)
print("average validation zero satisfaction share:", average_validation_zero)
print("average validation worst-member satisfaction:", average_validation_worst)
print("nash validation NDCG@10:", nash_validation_ndcg)

lambda_tuning_rows = []

for lambda_std in lambda_grid:
    lambda_df = evaluate_mf_aggregation_on_validation(
        aggregation="tuned_fairness",
        lambda_std=lambda_std,
    )

    lambda_tuning_rows.append(
        {
            "lambda_std": float(lambda_std),
            "mean_group_ndcg_10": float(lambda_df["group_ndcg_10"].mean()),
            "mean_jain_fairness": float(lambda_df["jain_fairness"].mean()),
            "mean_worst_member_satisfaction": float(lambda_df["worst_member_satisfaction"].mean()),
            "mean_zero_satisfaction_share": float(lambda_df["zero_satisfaction_share"].mean()),
            "mean_member_hit_share": float(lambda_df["member_hit_share"].mean()),
        }
    )

lambda_tuning_df = pd.DataFrame(lambda_tuning_rows)

# Fairness-first selection with a utility floor:
# accept up to 10% NDCG loss compared with average aggregation,
# then prefer better fairness.
lambda_ndcg_floor = 0.90 * average_validation_ndcg

lambda_feasible = lambda_tuning_df[
    lambda_tuning_df["mean_group_ndcg_10"] >= lambda_ndcg_floor
].copy()

if len(lambda_feasible) == 0:
    lambda_feasible = lambda_tuning_df.copy()
    lambda_selection_rule = (
        "fallback: no lambda met the 10% NDCG floor; "
        "selected best fairness over all lambda values"
    )
else:
    lambda_selection_rule = (
        "fairness-first: max Jain fairness and lower zero-satisfaction "
        "subject to validation NDCG within 10% of average aggregation"
    )

best_lambda_row = lambda_feasible.sort_values(
    by=[
        "mean_jain_fairness",
        "mean_zero_satisfaction_share",
        "mean_worst_member_satisfaction",
        "mean_group_ndcg_10",
    ],
    ascending=[False, True, False, False],
).iloc[0]

selected_lambda_std = float(best_lambda_row["lambda_std"])

validation_diverse_minority_groups = [
    group for group in validation_groups
    if group["group_type"] == "diverse_minority"
]

alpha_tuning_rows = []

for alpha_minority in alpha_grid:
    rows = []

    for group in validation_diverse_minority_groups:
        eligible_items, top_item_idx, _ = score_group_with_matrix(
            group=group,
            score_matrix=mf_score_matrix,
            seen_by_user_idx=train_seen_by_user_idx,
            aggregation="dbscan_minority_nash",
            alpha_minority=alpha_minority,
        )

        metric_row, _ = evaluate_group_recommendation(
            group=group,
            recommended_items=top_item_idx,
            eligible_items=eligible_items,
            relevant_by_user_idx=validation_relevant_by_user_idx,
        )

        rows.append(metric_row)

    alpha_df = pd.DataFrame(rows)

    alpha_tuning_rows.append(
        {
            "alpha_minority": float(alpha_minority),
            "mean_group_ndcg_10": float(alpha_df["group_ndcg_10"].mean()),
            "mean_jain_fairness": float(alpha_df["jain_fairness"].mean()),
            "mean_minority_cluster_satisfaction": float(alpha_df["minority_cluster_satisfaction"].mean()),
            "mean_minority_zero_satisfaction_rate": float(alpha_df["minority_zero_satisfaction_rate"].mean()),
            "mean_worst_member_satisfaction": float(alpha_df["worst_member_satisfaction"].mean()),
            "mean_zero_satisfaction_share": float(alpha_df["zero_satisfaction_share"].mean()),
            "mean_member_hit_share": float(alpha_df["member_hit_share"].mean()),
        }
    )

alpha_tuning_df = pd.DataFrame(alpha_tuning_rows)

# Minority-aware method should be allowed a larger utility trade-off than plain Nash.
# Therefore use a 10% NDCG floor instead of the previous 5%.
alpha_ndcg_floor = 0.90 * nash_validation_ndcg

alpha_feasible = alpha_tuning_df[
    alpha_tuning_df["mean_group_ndcg_10"] >= alpha_ndcg_floor
].copy()

if len(alpha_feasible) == 0:
    alpha_feasible = alpha_tuning_df.copy()
    alpha_selection_rule = (
        "fallback: no alpha met the 10% NDCG floor; "
        "selected best minority satisfaction over all alpha values"
    )
else:
    alpha_selection_rule = (
        "minority-first: max minority-cluster satisfaction and lower minority zero-rate "
        "subject to validation NDCG within 10% of Nash"
    )

best_alpha_row = alpha_feasible.sort_values(
    by=[
        "mean_minority_cluster_satisfaction",
        "mean_minority_zero_satisfaction_rate",
        "mean_jain_fairness",
        "mean_group_ndcg_10",
    ],
    ascending=[False, True, False, False],
).iloc[0]

selected_alpha_minority = float(best_alpha_row["alpha_minority"])

validation_tuning_summary_df = pd.concat(
    [
        lambda_tuning_df.assign(tuning_parameter="lambda_std"),
        alpha_tuning_df.assign(tuning_parameter="alpha_minority"),
    ],
    ignore_index=True,
    sort=False,
)

tuning_decision = {
    "top_k": int(top_k),
    "lambda_grid": [float(value) for value in lambda_grid],
    "alpha_grid": [float(value) for value in alpha_grid],
    "average_validation_ndcg_10": average_validation_ndcg,
    "average_validation_jain": average_validation_jain,
    "average_validation_zero_satisfaction_share": average_validation_zero,
    "average_validation_worst_member_satisfaction": average_validation_worst,
    "nash_validation_ndcg_10": nash_validation_ndcg,
    "selected_lambda_std": selected_lambda_std,
    "lambda_selection_rule": lambda_selection_rule,
    "selected_alpha_minority": selected_alpha_minority,
    "alpha_selection_rule": alpha_selection_rule,
    "normalization": "per-member min-max normalization before group aggregation",
    "dbscan_minority_nash": (
        "normalized Nash score + alpha * normalized minority score "
        "+ 0.25 * alpha * normalized worst-member score"
    ),
}

validation_tuning_summary_df.to_csv(step5_6_dir / "step5_validation_tuning_summary.csv", index=False)
validation_tuning_summary_df.to_parquet(step5_6_dir / "step5_validation_tuning_summary.parquet", index=False)

with open(step5_6_dir / "step5_validation_tuning_decision.json", "w") as file:
    json.dump(tuning_decision, file, indent=2)

print("selected lambda_std:", selected_lambda_std)
print("lambda selection rule:", lambda_selection_rule)
print("selected alpha_minority:", selected_alpha_minority)
print("alpha selection rule:", alpha_selection_rule)

display(lambda_tuning_df)
display(alpha_tuning_df)

average validation NDCG@10: 0.05701876755002145
average validation Jain: 0.15056758572173504
average validation zero satisfaction share: 0.845
average validation worst-member satisfaction: 0.0015120191524692545
nash validation NDCG@10: 0.0566432072477378
selected lambda_std: 0.0
lambda selection rule: fairness-first: max Jain fairness and lower zero-satisfaction subject to validation NDCG within 10% of average aggregation
selected alpha_minority: 2.0
alpha selection rule: minority-first: max minority-cluster satisfaction and lower minority zero-rate subject to validation NDCG within 10% of Nash


,lambda_std,mean_group_ndcg_10,mean_jain_fairness,mean_worst_member_satisfaction,mean_zero_satisfaction_share,mean_member_hit_share
0,0.00,0.057019,0.150568,0.001512,0.845000,0.155000
1,0.25,0.057056,0.147270,0.001512,0.848333,0.151667
2,0.50,0.055258,0.145018,0.001416,0.851667,0.148333
3,1.00,0.052771,0.146369,0.001320,0.850000,0.150000
4,2.00,0.049749,0.140032,0.000868,0.856667,0.143333
5,4.00,0.037267,0.108921,0.000000,0.888333,0.111667


,alpha_minority,mean_group_ndcg_10,mean_jain_fairness,mean_minority_cluster_satisfaction,mean_minority_zero_satisfaction_rate,mean_worst_member_satisfaction,mean_zero_satisfaction_share,mean_member_hit_share
0,0.00,0.070785,0.165625,0.025256,0.88,0.0,0.833333,0.166667
1,0.25,0.053909,0.136348,0.021753,0.86,0.0,0.860000,0.140000
2,0.50,0.054990,0.150357,0.030787,0.80,0.0,0.846667,0.153333
3,1.00,0.056810,0.157122,0.031396,0.80,0.0,0.840000,0.160000
4,2.00,0.057986,0.157122,0.034328,0.78,0.0,0.840000,0.160000
5,4.00,0.059255,0.163788,0.034286,0.78,0.0,0.833333,0.166667


# 6 Bayesian helpers for the hybrid MF + Bayesian score source

In [16]:
def build_pair_features(user_idx, first_item_idx, second_item_idx, feature_set):
    user_idx = np.asarray(user_idx, dtype=np.int32)
    first_item_idx = np.asarray(first_item_idx, dtype=np.int32)
    second_item_idx = np.asarray(second_item_idx, dtype=np.int32)

    p_u = user_embeddings[user_idx]
    q_first = item_embeddings[first_item_idx]
    q_second = item_embeddings[second_item_idx]
    q_diff = q_first - q_second

    if feature_set == "user_vector":
        features = p_u
    elif feature_set == "item_vectors":
        features = np.concatenate([q_first, q_second], axis=1)
    elif feature_set == "item_difference":
        features = q_diff
    elif feature_set == "user_item_interactions":
        features = np.concatenate([p_u * q_first, p_u * q_second], axis=1)
    elif feature_set == "personalized_difference":
        features = np.concatenate([p_u * q_diff, q_diff], axis=1)
    elif feature_set == "full_embedding_features":
        features = np.concatenate(
            [
                p_u,
                q_first,
                q_second,
                q_diff,
                p_u * q_first,
                p_u * q_second,
                p_u * q_diff,
            ],
            axis=1,
        )
    else:
        raise ValueError(f"unknown feature set: {feature_set}")

    return features.astype(np.float32)


def predict_bayes_prob_arrays(user_idx, first_item_idx, second_item_idx, artifact, chunk_size=100_000):
    model = artifact["model"]
    discretizer = artifact["discretizer"]
    feature_set = artifact["feature_set"]

    class_list = list(model.classes_)

    if 1 not in class_list:
        raise ValueError("bayesian model does not contain class 1")

    prob_col = class_list.index(1)
    probabilities = np.zeros(len(user_idx), dtype=np.float32)

    for start in range(0, len(user_idx), chunk_size):
        end = min(start + chunk_size, len(user_idx))

        x_part = build_pair_features(
            user_idx=user_idx[start:end],
            first_item_idx=first_item_idx[start:end],
            second_item_idx=second_item_idx[start:end],
            feature_set=feature_set,
        )

        x_part_bin = discretizer.transform(x_part).astype(np.int16)
        probabilities[start:end] = model.predict_proba(x_part_bin)[:, prob_col]

    return probabilities


def get_bayes_borda_scores_for_user(user_idx, candidate_item_idx, artifact):
    candidate_item_idx = np.asarray(candidate_item_idx, dtype=np.int32)

    if len(candidate_item_idx) != len(np.unique(candidate_item_idx)):
        raise ValueError("candidate_item_idx contains duplicated items")

    candidate_count = len(candidate_item_idx)

    if candidate_count < 2:
        return np.ones(candidate_count, dtype=np.float32)

    first_positions = np.repeat(np.arange(candidate_count), candidate_count)
    second_positions = np.tile(np.arange(candidate_count), candidate_count)
    mask = first_positions != second_positions

    first_positions = first_positions[mask]
    second_positions = second_positions[mask]

    first_item_idx = candidate_item_idx[first_positions]
    second_item_idx = candidate_item_idx[second_positions]
    user_idx_array = np.full(len(first_item_idx), int(user_idx), dtype=np.int32)

    # Raw Naive Bayes probabilities are not guaranteed to be antisymmetric.
    # Step 4 evaluates Bayesian probabilities with symmetric correction:
    # P_sym(i > j) = 0.5 * (P(i > j) + 1 - P(j > i)).
    # We apply the same correction here before computing Borda scores.
    raw_probabilities = predict_bayes_prob_arrays(
        user_idx=user_idx_array,
        first_item_idx=first_item_idx,
        second_item_idx=second_item_idx,
        artifact=artifact,
    )

    probability_matrix = np.full(
        (candidate_count, candidate_count),
        np.nan,
        dtype=np.float32,
    )
    probability_matrix[first_positions, second_positions] = raw_probabilities

    symmetric_probability_matrix = 0.5 * (
        probability_matrix + 1.0 - probability_matrix.T
    )

    np.fill_diagonal(symmetric_probability_matrix, np.nan)

    off_diagonal_mask = ~np.eye(candidate_count, dtype=bool)
    if np.isnan(symmetric_probability_matrix[off_diagonal_mask]).any():
        raise ValueError("symmetric Bayesian probability matrix contains missing off-diagonal values")

    symmetric_probability_matrix = np.clip(
        symmetric_probability_matrix,
        1e-12,
        1.0 - 1e-12,
    )

    score_sum = np.nansum(symmetric_probability_matrix, axis=1)
    score_count = np.sum(~np.isnan(symmetric_probability_matrix), axis=1)

    return (score_sum / np.maximum(score_count, 1)).astype(np.float32)


def get_bayes_member_score_matrix(user_idx_list, candidate_item_idx, artifact):
    candidate_item_idx = np.asarray(candidate_item_idx, dtype=np.int32)

    if len(candidate_item_idx) != len(np.unique(candidate_item_idx)):
        raise ValueError("hybrid candidate list contains duplicated items")

    member_scores = []

    for user_idx in user_idx_list:
        member_scores.append(
            get_bayes_borda_scores_for_user(
                user_idx=int(user_idx),
                candidate_item_idx=candidate_item_idx,
                artifact=artifact,
            )
        )

    member_scores = np.vstack(member_scores).astype(np.float32)
    assert member_scores.shape[1] == len(candidate_item_idx)

    return member_scores

# 7 Generate final test recommendations

In [17]:
aggregation_methods = [
    "average",
    "least_misery",
    "tuned_fairness",
    "nash_welfare",
    "dbscan_minority_nash",
]

score_sources = {
    "mf": mf_score_matrix,
    "rating_user_knn": knn_score_matrix,
}


def make_recommendation_rows(group, method_name, score_source, aggregation, item_idx_list, group_score_list):
    rows = []

    for rank, (item_idx, group_score) in enumerate(zip(item_idx_list, group_score_list), start=1):
        movie_id = int(item_idx_to_movie_id[int(item_idx)])

        rows.append(
            {
                "group_id": group["group_id"],
                "group_type": group["group_type"],
                "method": method_name,
                "score_source": score_source,
                "aggregation": aggregation,
                "rank": rank,
                "item_idx": int(item_idx),
                "movie_id": movie_id,
                "title": movie_id_to_title.get(movie_id, str(movie_id)),
                "group_score": float(group_score),
                "lambda_std": selected_lambda_std if aggregation == "tuned_fairness" else np.nan,
                "alpha_minority": selected_alpha_minority if aggregation == "dbscan_minority_nash" else np.nan,
            }
        )

    return rows



final_test_groups = test_groups
groups = final_test_groups

recommendation_rows = []

for group_number, group in enumerate(final_test_groups, start=1):
    if group_number % 10 == 0:
        print(f"processing final test group {group_number}/{len(final_test_groups)}")

    user_idx_list = group["user_idx_list"]
    eligible_item_idx = get_eligible_items_for_group(user_idx_list, train_validation_seen_by_user_idx)

    if len(eligible_item_idx) < top_k:
        raise ValueError(f"group {group['group_id']} has too few eligible test candidates")

    for score_source, score_matrix in score_sources.items():
        member_scores = score_matrix[np.asarray(user_idx_list, dtype=np.int32)][:, eligible_item_idx]

        for aggregation in aggregation_methods:
            group_scores = aggregate_member_scores(
                member_scores=member_scores,
                aggregation=aggregation,
                member_clusters=group["cluster_list"],
                minority_cluster=group["minority_cluster"],
                lambda_std=selected_lambda_std,
                alpha_minority=selected_alpha_minority,
            )

            top_item_idx, top_scores = get_top_items_from_scores(
                item_idx_array=eligible_item_idx,
                score_array=group_scores,
                top_n=top_k,
            )

            recommendation_rows.extend(
                make_recommendation_rows(
                    group=group,
                    method_name=f"{score_source}_{aggregation}",
                    score_source=score_source,
                    aggregation=aggregation,
                    item_idx_list=top_item_idx,
                    group_score_list=top_scores,
                )
            )

    # Hybrid MF + Bayesian: MF average selects a manageable candidate set, Bayesian Borda provides member scores.
    mf_member_scores = mf_score_matrix[np.asarray(user_idx_list, dtype=np.int32)][:, eligible_item_idx]
    mf_average_scores = aggregate_member_scores(
        member_scores=mf_member_scores,
        aggregation="average",
    )

    hybrid_candidate_item_idx, _ = get_top_items_from_scores(
        item_idx_array=eligible_item_idx,
        score_array=mf_average_scores,
        top_n=hybrid_rerank_size,
    )

    bayes_member_scores = get_bayes_member_score_matrix(
        user_idx_list=user_idx_list,
        candidate_item_idx=hybrid_candidate_item_idx,
        artifact=bayes_artifact,
    )

    for aggregation in aggregation_methods:
        group_scores = aggregate_member_scores(
            member_scores=bayes_member_scores,
            aggregation=aggregation,
            member_clusters=group["cluster_list"],
            minority_cluster=group["minority_cluster"],
            lambda_std=selected_lambda_std,
            alpha_minority=selected_alpha_minority,
        )

        top_item_idx, top_scores = get_top_items_from_scores(
            item_idx_array=hybrid_candidate_item_idx,
            score_array=group_scores,
            top_n=top_k,
        )

        recommendation_rows.extend(
            make_recommendation_rows(
                group=group,
                method_name=f"hybrid_mf_bayes_{aggregation}",
                score_source="hybrid_mf_bayes",
                aggregation=aggregation,
                item_idx_list=top_item_idx,
                group_score_list=top_scores,
            )
        )

recommendations_df = pd.DataFrame(recommendation_rows)

display(recommendations_df.head())
display(recommendations_df.groupby(["score_source", "aggregation"]).size().reset_index(name="rows"))
print("recommendation rows:", len(recommendations_df))

processing final test group 10/200
processing final test group 20/200
processing final test group 30/200
processing final test group 40/200
processing final test group 50/200
processing final test group 60/200
processing final test group 70/200
processing final test group 80/200
processing final test group 90/200
processing final test group 100/200
processing final test group 110/200
processing final test group 120/200
processing final test group 130/200
processing final test group 140/200
processing final test group 150/200
processing final test group 160/200
processing final test group 170/200
processing final test group 180/200
processing final test group 190/200
processing final test group 200/200


,group_id,group_type,method,score_source,aggregation,rank,item_idx,movie_id,title,group_score,lambda_std,alpha_minority
0,test_group_0001,homogeneous_cluster_0,mf_average,mf,average,1,283,318,"Shawshank Redemption, The (1994)",0.842499,NaN,NaN
1,test_group_0001,homogeneous_cluster_0,mf_average,mf,average,2,2381,2905,Sanjuro (1962),0.790056,NaN,NaN
2,test_group_0001,homogeneous_cluster_0,mf_average,mf,average,3,1012,1262,"Great Escape, The (1963)",0.787707,NaN,NaN
3,test_group_0001,homogeneous_cluster_0,mf_average,mf,average,4,1605,2019,Seven Samurai (The Magnificent Seven) (Shichin...,0.781438,NaN,NaN
4,test_group_0001,homogeneous_cluster_0,mf_average,mf,average,5,717,904,Rear Window (1954),0.753453,NaN,NaN


,score_source,aggregation,rows
0,hybrid_mf_bayes,average,2000
1,hybrid_mf_bayes,dbscan_minority_nash,2000
2,hybrid_mf_bayes,least_misery,2000
3,hybrid_mf_bayes,nash_welfare,2000
4,hybrid_mf_bayes,tuned_fairness,2000
5,mf,average,2000
6,mf,dbscan_minority_nash,2000
7,mf,least_misery,2000
8,mf,nash_welfare,2000
9,mf,tuned_fairness,2000


recommendation rows: 30000


In [18]:
group_lookup = {group["group_id"]: group for group in groups}

expected_methods = len(
    recommendations_df[["score_source", "aggregation", "method"]]
    .drop_duplicates()
)
expected_rows = len(groups) * expected_methods * top_k

assert len(recommendations_df) == expected_rows, f"expected {expected_rows} recommendation rows, got {len(recommendations_df)}"

list_sizes = recommendations_df.groupby(["group_id", "method"]).size()
assert list_sizes.eq(top_k).all(), "some group-method lists do not have exactly top_k items"

unique_item_counts = recommendations_df.groupby(["group_id", "method"])["item_idx"].nunique()
assert unique_item_counts.eq(top_k).all(), "some group-method lists contain duplicated items"

seen_violations = []

for group_id, rec_part in recommendations_df.groupby("group_id"):
    group = group_lookup[group_id]
    seen_items = set()

    for user_idx in group["user_idx_list"]:
        seen_items.update(train_validation_seen_by_user_idx.get(int(user_idx), set()))

    recommended_items = set(rec_part["item_idx"].astype(int).tolist())
    overlap = recommended_items.intersection(seen_items)

    if len(overlap) > 0:
        seen_violations.append({"group_id": group_id, "seen_recommended_count": len(overlap)})

seen_violations_df = pd.DataFrame(seen_violations)
assert len(seen_violations_df) == 0, "some recommendations contain train+validation seen items"

print("methods:", expected_methods)
print("rows:", len(recommendations_df))

methods: 15
rows: 30000


In [19]:
recommendations_df.to_csv(step5_6_dir / "step5_group_recommendations.csv", index=False)
recommendations_df.to_parquet(step5_6_dir / "step5_group_recommendations.parquet", index=False)

print("saved step 5 recommendation outputs")

saved step 5 recommendation outputs


# 8 Step 6 individual ranking evaluation on all eligible users

In [20]:
individual_user_idx = sorted(
    set(test_relevant_by_user_idx.keys()).intersection(train_validation_seen_by_user_idx.keys())
)

individual_metric_rows = []

for user_number, user_idx in enumerate(individual_user_idx, start=1):
    if user_number % 500 == 0:
        print(f"evaluating individual user {user_number}/{len(individual_user_idx)}")

    seen_items = train_validation_seen_by_user_idx.get(int(user_idx), set())
    relevant_items = test_relevant_by_user_idx.get(int(user_idx), set())

    eligible_items = np.array(
        [item_idx for item_idx in all_item_idx if int(item_idx) not in seen_items],
        dtype=np.int32,
    )

    if len(eligible_items) < top_k:
        continue

    for score_source, score_matrix in [
        ("mf", mf_score_matrix),
        ("rating_user_knn", knn_score_matrix),
    ]:
        user_scores = score_matrix[int(user_idx), eligible_items]

        top_item_idx, _ = get_top_items_from_scores(
            item_idx_array=eligible_items,
            score_array=user_scores,
            top_n=top_k,
        )

        individual_metric_rows.append(
            {
                "user_idx": int(user_idx),
                "user_id": int(user_idx_to_id[int(user_idx)]),
                "user_cluster": int(user_idx_to_cluster.get(int(user_idx), -999)),
                "score_source": score_source,
                "precision_10": precision_at_k(top_item_idx, relevant_items, top_k),
                "recall_10": recall_at_k(top_item_idx, relevant_items, top_k),
                "hit_rate_10": hit_rate_at_k(top_item_idx, relevant_items, top_k),
                "ndcg_10": ndcg_at_k_binary(top_item_idx, relevant_items, top_k),
            }
        )

individual_metrics_df = pd.DataFrame(individual_metric_rows)

individual_summary_df = (
    individual_metrics_df
    .groupby("score_source", as_index=False)
    .agg(
        users=("user_idx", "nunique"),
        mean_precision_10=("precision_10", "mean"),
        mean_recall_10=("recall_10", "mean"),
        mean_hit_rate_10=("hit_rate_10", "mean"),
        mean_ndcg_10=("ndcg_10", "mean"),
    )
    .sort_values("mean_ndcg_10", ascending=False)
    .reset_index(drop=True)
)

display(individual_summary_df)

evaluating individual user 500/4623
evaluating individual user 1000/4623
evaluating individual user 1500/4623
evaluating individual user 2000/4623
evaluating individual user 2500/4623
evaluating individual user 3000/4623
evaluating individual user 3500/4623
evaluating individual user 4000/4623
evaluating individual user 4500/4623


,score_source,users,mean_precision_10,mean_recall_10,mean_hit_rate_10,mean_ndcg_10
0,mf,4623,0.037184,0.034616,0.241834,0.043403
1,rating_user_knn,4623,0.005278,0.004357,0.048237,0.006232


# 9 Step 6 group recommendation and fairness evaluation

In [21]:
group_metric_rows = []
member_metric_rows = []

group_columns = [
    "group_id",
    "group_type",
    "method",
    "score_source",
    "aggregation",
]

for key_values, rec_part in recommendations_df.groupby(group_columns):
    key_dict = dict(zip(group_columns, key_values))
    group = group_lookup[key_dict["group_id"]]
    user_idx_list = group["user_idx_list"]

    rec_part = rec_part.sort_values("rank")
    recommended_items = rec_part["item_idx"].astype(int).tolist()
    eligible_item_idx = get_eligible_items_for_group(user_idx_list, train_validation_seen_by_user_idx)

    metric_row, member_rows = evaluate_group_recommendation(
        group=group,
        recommended_items=recommended_items,
        eligible_items=eligible_item_idx,
        relevant_by_user_idx=test_relevant_by_user_idx,
    )

    group_metric_rows.append(
        {
            **key_dict,
            **metric_row,
            "cluster_entropy": group["cluster_entropy"],
            "avg_pairwise_embedding_distance": group["avg_pairwise_embedding_distance"],
            "majority_cluster": group["majority_cluster"],
            "minority_cluster": group["minority_cluster"],
            "noise_share": group["noise_share"],
            "lambda_std": selected_lambda_std if key_dict["aggregation"] == "tuned_fairness" else np.nan,
            "alpha_minority": selected_alpha_minority if key_dict["aggregation"] == "dbscan_minority_nash" else np.nan,
        }
    )

    for member_row in member_rows:
        member_metric_rows.append({**key_dict, **member_row})

group_metrics_df = pd.DataFrame(group_metric_rows)
member_metrics_df = pd.DataFrame(member_metric_rows)

display(group_metrics_df.head())
display(member_metrics_df.head())

,group_id,group_type,method,score_source,aggregation,group_precision_10,group_recall_10,group_ndcg_10,mean_member_ndcg_10,mean_satisfaction,std_satisfaction,worst_member_satisfaction,jain_fairness,minority_cluster_satisfaction,minority_zero_satisfaction_rate,cluster_satisfaction_gap,zero_satisfaction_share,member_hit_share,intra_list_embedding_distance,cluster_entropy,avg_pairwise_embedding_distance,majority_cluster,minority_cluster,noise_share,lambda_std,alpha_minority
0,test_group_0001,homogeneous_cluster_0,hybrid_mf_bayes_average,hybrid_mf_bayes,average,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,1.0,0.0,0.311397,0.0,1.055191,NaN,NaN,0.0,NaN,NaN
1,test_group_0001,homogeneous_cluster_0,hybrid_mf_bayes_dbscan_minority_nash,hybrid_mf_bayes,dbscan_minority_nash,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,1.0,0.0,0.311397,0.0,1.055191,NaN,NaN,0.0,NaN,2.0
2,test_group_0001,homogeneous_cluster_0,hybrid_mf_bayes_least_misery,hybrid_mf_bayes,least_misery,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,1.0,0.0,0.267518,0.0,1.055191,NaN,NaN,0.0,NaN,NaN
3,test_group_0001,homogeneous_cluster_0,hybrid_mf_bayes_nash_welfare,hybrid_mf_bayes,nash_welfare,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,1.0,0.0,0.311397,0.0,1.055191,NaN,NaN,0.0,NaN,NaN
4,test_group_0001,homogeneous_cluster_0,hybrid_mf_bayes_tuned_fairness,hybrid_mf_bayes,tuned_fairness,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,1.0,0.0,0.311397,0.0,1.055191,NaN,NaN,0.0,0.0,NaN


,group_id,group_type,method,score_source,aggregation,user_idx,user_id,user_cluster,member_precision_10,member_recall_10,member_ndcg_10,member_hit_10,member_satisfaction
0,test_group_0001,homogeneous_cluster_0,hybrid_mf_bayes_average,hybrid_mf_bayes,average,4411,5648,0,0.0,0.0,0.0,0.0,0.0
1,test_group_0001,homogeneous_cluster_0,hybrid_mf_bayes_average,hybrid_mf_bayes,average,2090,2698,0,0.0,0.0,0.0,0.0,0.0
2,test_group_0001,homogeneous_cluster_0,hybrid_mf_bayes_average,hybrid_mf_bayes,average,2795,3599,0,0.0,0.0,0.0,0.0,0.0
3,test_group_0001,homogeneous_cluster_0,hybrid_mf_bayes_dbscan_minority_nash,hybrid_mf_bayes,dbscan_minority_nash,4411,5648,0,0.0,0.0,0.0,0.0,0.0
4,test_group_0001,homogeneous_cluster_0,hybrid_mf_bayes_dbscan_minority_nash,hybrid_mf_bayes,dbscan_minority_nash,2090,2698,0,0.0,0.0,0.0,0.0,0.0


### 9.1 Method, group type, minority, and diversity summaries

In [22]:
method_summary_df = (
    group_metrics_df
    .groupby(["score_source", "aggregation", "method"], as_index=False)
    .agg(
        groups=("group_id", "nunique"),
        mean_group_precision_10=("group_precision_10", "mean"),
        mean_group_recall_10=("group_recall_10", "mean"),
        mean_group_ndcg_10=("group_ndcg_10", "mean"),
        mean_member_ndcg_10=("mean_member_ndcg_10", "mean"),
        mean_worst_member_satisfaction=("worst_member_satisfaction", "mean"),
        mean_jain_fairness=("jain_fairness", "mean"),
        mean_minority_cluster_satisfaction=("minority_cluster_satisfaction", "mean"),
        mean_minority_zero_satisfaction_rate=("minority_zero_satisfaction_rate", "mean"),
        mean_cluster_satisfaction_gap=("cluster_satisfaction_gap", "mean"),
        mean_zero_satisfaction_share=("zero_satisfaction_share", "mean"),
        mean_member_hit_share=("member_hit_share", "mean"),
        mean_intra_list_embedding_distance=("intra_list_embedding_distance", "mean"),
    )
    .sort_values(by=["mean_group_ndcg_10", "mean_jain_fairness"], ascending=[False, False])
    .reset_index(drop=True)
)

group_type_summary_df = (
    group_metrics_df
    .groupby(["group_type", "score_source", "aggregation"], as_index=False)
    .agg(
        groups=("group_id", "nunique"),
        mean_group_ndcg_10=("group_ndcg_10", "mean"),
        mean_member_ndcg_10=("mean_member_ndcg_10", "mean"),
        mean_worst_member_satisfaction=("worst_member_satisfaction", "mean"),
        mean_jain_fairness=("jain_fairness", "mean"),
        mean_minority_cluster_satisfaction=("minority_cluster_satisfaction", "mean"),
        mean_minority_zero_satisfaction_rate=("minority_zero_satisfaction_rate", "mean"),
        mean_cluster_satisfaction_gap=("cluster_satisfaction_gap", "mean"),
        mean_zero_satisfaction_share=("zero_satisfaction_share", "mean"),
        mean_member_hit_share=("member_hit_share", "mean"),
    )
    .sort_values(by=["group_type", "mean_group_ndcg_10"], ascending=[True, False])
    .reset_index(drop=True)
)

minority_summary_df = (
    group_metrics_df
    .loc[group_metrics_df["group_type"] == "diverse_minority"]
    .groupby(["score_source", "aggregation", "method"], as_index=False)
    .agg(
        groups=("group_id", "nunique"),
        mean_group_ndcg_10=("group_ndcg_10", "mean"),
        mean_minority_cluster_satisfaction=("minority_cluster_satisfaction", "mean"),
        mean_minority_zero_satisfaction_rate=("minority_zero_satisfaction_rate", "mean"),
        mean_cluster_satisfaction_gap=("cluster_satisfaction_gap", "mean"),
        mean_jain_fairness=("jain_fairness", "mean"),
        mean_member_hit_share=("member_hit_share", "mean"),
        mean_zero_satisfaction_share=("zero_satisfaction_share", "mean"),
    )
    .sort_values(
        by=["mean_minority_cluster_satisfaction", "mean_minority_zero_satisfaction_rate", "mean_group_ndcg_10"],
        ascending=[False, True, False],
    )
    .reset_index(drop=True)
)

diversity_metrics_df = group_metrics_df.copy()
diversity_metrics_df["embedding_distance_bin"] = pd.qcut(
    diversity_metrics_df["avg_pairwise_embedding_distance"],
    q=3,
    labels=["low", "medium", "high"],
    duplicates="drop",
)

diversity_summary_df = (
    diversity_metrics_df
    .groupby(["embedding_distance_bin", "score_source", "aggregation"], as_index=False, observed=False)
    .agg(
        groups=("group_id", "nunique"),
        mean_group_ndcg_10=("group_ndcg_10", "mean"),
        mean_member_ndcg_10=("mean_member_ndcg_10", "mean"),
        mean_worst_member_satisfaction=("worst_member_satisfaction", "mean"),
        mean_jain_fairness=("jain_fairness", "mean"),
    )
    .sort_values(by=["embedding_distance_bin", "mean_group_ndcg_10"], ascending=[True, False])
    .reset_index(drop=True)
)

display(method_summary_df)
display(group_type_summary_df.head(30))
display(minority_summary_df)
display(diversity_summary_df)

,score_source,aggregation,method,groups,mean_group_precision_10,mean_group_recall_10,mean_group_ndcg_10,mean_member_ndcg_10,mean_worst_member_satisfaction,mean_jain_fairness,mean_minority_cluster_satisfaction,mean_minority_zero_satisfaction_rate,mean_cluster_satisfaction_gap,mean_zero_satisfaction_share,mean_member_hit_share,mean_intra_list_embedding_distance
0,mf,nash_welfare,mf_nash_welfare,200,0.022833,0.028766,0.067655,0.031071,0.000000,0.158256,0.035309,0.82,0.032037,0.838333,0.161667,0.355482
1,mf,average,mf_average,200,0.022333,0.027725,0.066306,0.030300,0.000000,0.154878,0.035110,0.82,0.031528,0.841667,0.158333,0.355737
2,mf,tuned_fairness,mf_tuned_fairness,200,0.022333,0.027725,0.066306,0.030300,0.000000,0.154878,0.035110,0.82,0.031528,0.841667,0.158333,0.355737
3,mf,dbscan_minority_nash,mf_dbscan_minority_nash,200,0.022333,0.027800,0.065524,0.030383,0.000000,0.156630,0.030297,0.84,0.030032,0.840000,0.160000,0.358922
4,hybrid_mf_bayes,nash_welfare,hybrid_mf_bayes_nash_welfare,200,0.020667,0.024686,0.061982,0.027303,0.000000,0.141837,0.030807,0.86,0.025500,0.855000,0.145000,0.331598
5,hybrid_mf_bayes,dbscan_minority_nash,hybrid_mf_bayes_dbscan_minority_nash,200,0.020667,0.024543,0.061871,0.027404,0.000000,0.141285,0.027178,0.86,0.025671,0.855000,0.145000,0.330672
6,hybrid_mf_bayes,average,hybrid_mf_bayes_average,200,0.020500,0.024045,0.060999,0.027200,0.000000,0.142840,0.032962,0.84,0.026046,0.853333,0.146667,0.326010
7,hybrid_mf_bayes,tuned_fairness,hybrid_mf_bayes_tuned_fairness,200,0.020500,0.024045,0.060999,0.027200,0.000000,0.142840,0.032962,0.84,0.026046,0.853333,0.146667,0.326010
8,hybrid_mf_bayes,least_misery,hybrid_mf_bayes_least_misery,200,0.019333,0.024409,0.058098,0.026616,0.000000,0.138048,0.029591,0.88,0.025551,0.858333,0.141667,0.342283
9,mf,least_misery,mf_least_misery,200,0.018167,0.021269,0.055390,0.026466,0.000331,0.123966,0.028470,0.90,0.028021,0.871667,0.128333,0.362606


,group_type,score_source,aggregation,groups,mean_group_ndcg_10,mean_member_ndcg_10,mean_worst_member_satisfaction,mean_jain_fairness,mean_minority_cluster_satisfaction,mean_minority_zero_satisfaction_rate,mean_cluster_satisfaction_gap,mean_zero_satisfaction_share,mean_member_hit_share
0,diverse_balanced,mf,dbscan_minority_nash,50,0.048252,0.018975,0.000000,0.126576,NaN,NaN,0.053798,0.873333,0.126667
1,diverse_balanced,mf,nash_welfare,50,0.048252,0.018975,0.000000,0.126576,NaN,NaN,0.053798,0.873333,0.126667
2,diverse_balanced,mf,least_misery,50,0.048070,0.022175,0.001325,0.103742,NaN,NaN,0.055576,0.893333,0.106667
3,diverse_balanced,mf,average,50,0.045021,0.017358,0.000000,0.120000,NaN,NaN,0.052074,0.880000,0.120000
4,diverse_balanced,mf,tuned_fairness,50,0.045021,0.017358,0.000000,0.120000,NaN,NaN,0.052074,0.880000,0.120000
5,diverse_balanced,hybrid_mf_bayes,dbscan_minority_nash,50,0.035788,0.012972,0.000000,0.080000,NaN,NaN,0.038916,0.920000,0.080000
6,diverse_balanced,hybrid_mf_bayes,nash_welfare,50,0.035788,0.012972,0.000000,0.080000,NaN,NaN,0.038916,0.920000,0.080000
7,diverse_balanced,hybrid_mf_bayes,least_misery,50,0.034171,0.014226,0.000000,0.096361,NaN,NaN,0.041288,0.900000,0.100000
8,diverse_balanced,hybrid_mf_bayes,average,50,0.033417,0.013252,0.000000,0.076947,NaN,NaN,0.038052,0.920000,0.080000
9,diverse_balanced,hybrid_mf_bayes,tuned_fairness,50,0.033417,0.013252,0.000000,0.076947,NaN,NaN,0.038052,0.920000,0.080000


,score_source,aggregation,method,groups,mean_group_ndcg_10,mean_minority_cluster_satisfaction,mean_minority_zero_satisfaction_rate,mean_cluster_satisfaction_gap,mean_jain_fairness,mean_member_hit_share,mean_zero_satisfaction_share
0,mf,nash_welfare,mf_nash_welfare,50,0.088373,0.035309,0.82,0.074352,0.164506,0.166667,0.833333
1,mf,average,mf_average,50,0.088479,0.035110,0.82,0.074036,0.164057,0.166667,0.833333
2,mf,tuned_fairness,mf_tuned_fairness,50,0.088479,0.035110,0.82,0.074036,0.164057,0.166667,0.833333
3,hybrid_mf_bayes,average,hybrid_mf_bayes_average,50,0.079944,0.032962,0.84,0.066131,0.146017,0.146667,0.853333
4,hybrid_mf_bayes,tuned_fairness,hybrid_mf_bayes_tuned_fairness,50,0.079944,0.032962,0.84,0.066131,0.146017,0.146667,0.853333
5,rating_user_knn,average,rating_user_knn_average,50,0.042404,0.031072,0.84,0.043497,0.106667,0.106667,0.893333
6,rating_user_knn,tuned_fairness,rating_user_knn_tuned_fairness,50,0.042404,0.031072,0.84,0.043497,0.106667,0.106667,0.893333
7,hybrid_mf_bayes,nash_welfare,hybrid_mf_bayes_nash_welfare,50,0.077853,0.030807,0.86,0.063083,0.133023,0.133333,0.866667
8,mf,dbscan_minority_nash,mf_dbscan_minority_nash,50,0.079848,0.030297,0.84,0.066329,0.158001,0.160000,0.840000
9,rating_user_knn,nash_welfare,rating_user_knn_nash_welfare,50,0.041612,0.030232,0.86,0.042520,0.100000,0.100000,0.900000


,embedding_distance_bin,score_source,aggregation,groups,mean_group_ndcg_10,mean_member_ndcg_10,mean_worst_member_satisfaction,mean_jain_fairness
0,low,mf,nash_welfare,67,0.071793,0.029579,0.000000,0.191376
1,low,mf,average,67,0.069935,0.028710,0.000000,0.186401
2,low,mf,tuned_fairness,67,0.069935,0.028710,0.000000,0.186401
3,low,mf,dbscan_minority_nash,67,0.068830,0.028347,0.000000,0.186400
4,low,hybrid_mf_bayes,average,67,0.066978,0.026765,0.000000,0.176173
5,low,hybrid_mf_bayes,tuned_fairness,67,0.066978,0.026765,0.000000,0.176173
6,low,hybrid_mf_bayes,nash_welfare,67,0.066883,0.026733,0.000000,0.176173
7,low,hybrid_mf_bayes,dbscan_minority_nash,67,0.063792,0.025543,0.000000,0.176173
8,low,hybrid_mf_bayes,least_misery,67,0.056391,0.022445,0.000000,0.158099
9,low,mf,least_misery,67,0.052803,0.021952,0.000000,0.163248


### 9.2 Fairness trade off vs average aggregation

In [23]:
baseline_df = group_metrics_df.loc[
    group_metrics_df["aggregation"] == "average",
    [
        "group_id",
        "score_source",
        "group_ndcg_10",
        "jain_fairness",
        "worst_member_satisfaction",
        "std_satisfaction",
        "minority_cluster_satisfaction",
        "minority_zero_satisfaction_rate",
        "cluster_satisfaction_gap",
        "zero_satisfaction_share",
        "member_hit_share",
    ],
].rename(
    columns={
        "group_ndcg_10": "average_group_ndcg_10",
        "jain_fairness": "average_jain_fairness",
        "worst_member_satisfaction": "average_worst_member_satisfaction",
        "std_satisfaction": "average_std_satisfaction",
        "minority_cluster_satisfaction": "average_minority_cluster_satisfaction",
        "minority_zero_satisfaction_rate": "average_minority_zero_satisfaction_rate",
        "cluster_satisfaction_gap": "average_cluster_satisfaction_gap",
        "zero_satisfaction_share": "average_zero_satisfaction_share",
        "member_hit_share": "average_member_hit_share",
    }
)

tradeoff_df = group_metrics_df.loc[
    group_metrics_df["aggregation"].isin(["tuned_fairness", "nash_welfare", "dbscan_minority_nash"])
].merge(
    baseline_df,
    on=["group_id", "score_source"],
    how="left",
)

for metric in [
    "group_ndcg_10",
    "jain_fairness",
    "worst_member_satisfaction",
    "std_satisfaction",
    "minority_cluster_satisfaction",
    "minority_zero_satisfaction_rate",
    "cluster_satisfaction_gap",
    "zero_satisfaction_share",
    "member_hit_share",
]:
    tradeoff_df[f"delta_{metric}"] = tradeoff_df[metric] - tradeoff_df[f"average_{metric}"]

tradeoff_summary_df = (
    tradeoff_df
    .groupby(["score_source", "aggregation"], as_index=False)
    .agg(
        groups=("group_id", "nunique"),
        mean_delta_group_ndcg_10=("delta_group_ndcg_10", "mean"),
        mean_delta_jain_fairness=("delta_jain_fairness", "mean"),
        mean_delta_worst_member_satisfaction=("delta_worst_member_satisfaction", "mean"),
        mean_delta_std_satisfaction=("delta_std_satisfaction", "mean"),
        mean_delta_minority_cluster_satisfaction=("delta_minority_cluster_satisfaction", "mean"),
        mean_delta_minority_zero_satisfaction_rate=("delta_minority_zero_satisfaction_rate", "mean"),
        mean_delta_cluster_satisfaction_gap=("delta_cluster_satisfaction_gap", "mean"),
        mean_delta_zero_satisfaction_share=("delta_zero_satisfaction_share", "mean"),
        mean_delta_member_hit_share=("delta_member_hit_share", "mean"),
    )
    .sort_values(by=["score_source", "aggregation"])
    .reset_index(drop=True)
)

display(tradeoff_summary_df)

,score_source,aggregation,groups,mean_delta_group_ndcg_10,mean_delta_jain_fairness,mean_delta_worst_member_satisfaction,mean_delta_std_satisfaction,mean_delta_minority_cluster_satisfaction,mean_delta_minority_zero_satisfaction_rate,mean_delta_cluster_satisfaction_gap,mean_delta_zero_satisfaction_share,mean_delta_member_hit_share
0,hybrid_mf_bayes,dbscan_minority_nash,200,0.000872,-0.001554,0.0,-0.000178,-0.005784,0.02,-0.000375,0.001667,-0.001667
1,hybrid_mf_bayes,nash_welfare,200,0.000983,-0.001003,0.0,0.000055,-0.002155,0.02,-0.000546,0.001667,-0.001667
2,hybrid_mf_bayes,tuned_fairness,200,0.000000,0.000000,0.0,0.000000,0.000000,0.00,0.000000,0.000000,0.000000
3,mf,dbscan_minority_nash,200,-0.000782,0.001752,0.0,-0.001498,-0.004813,0.02,-0.001496,-0.001667,0.001667
4,mf,nash_welfare,200,0.001349,0.003378,0.0,0.000110,0.000199,0.00,0.000510,-0.003333,0.003333
5,mf,tuned_fairness,200,0.000000,0.000000,0.0,0.000000,0.000000,0.00,0.000000,0.000000,0.000000
6,rating_user_knn,dbscan_minority_nash,200,-0.004178,-0.006539,0.0,-0.002768,-0.018423,0.08,-0.004695,0.006667,-0.006667
7,rating_user_knn,nash_welfare,200,-0.000534,-0.001539,0.0,-0.000414,-0.000841,0.02,-0.000213,0.001667,-0.001667
8,rating_user_knn,tuned_fairness,200,0.000000,0.000000,0.0,0.000000,0.000000,0.00,0.000000,0.000000,0.000000


### 9.3 Counterexamples: best utility is not always best fairness

In [24]:
counterexample_rows = []

for group_id, group_part in group_metrics_df.groupby("group_id"):
    best_ndcg_row = group_part.loc[group_part["group_ndcg_10"].idxmax()]
    best_jain_row = group_part.loc[group_part["jain_fairness"].idxmax()]
    best_worst_row = group_part.loc[group_part["worst_member_satisfaction"].idxmax()]

    if best_ndcg_row["method"] != best_jain_row["method"]:
        counterexample_rows.append(
            {
                "group_id": group_id,
                "group_type": best_ndcg_row["group_type"],
                "fairness_metric": "jain_fairness",
                "best_ndcg_method": best_ndcg_row["method"],
                "best_fairness_method": best_jain_row["method"],
                "best_ndcg_value": float(best_ndcg_row["group_ndcg_10"]),
                "fair_method_ndcg_value": float(best_jain_row["group_ndcg_10"]),
                "best_ndcg_fairness_value": float(best_ndcg_row["jain_fairness"]),
                "fair_method_fairness_value": float(best_jain_row["jain_fairness"]),
                "ndcg_difference": float(best_ndcg_row["group_ndcg_10"] - best_jain_row["group_ndcg_10"]),
                "fairness_difference": float(best_jain_row["jain_fairness"] - best_ndcg_row["jain_fairness"]),
            }
        )

    if best_ndcg_row["method"] != best_worst_row["method"]:
        counterexample_rows.append(
            {
                "group_id": group_id,
                "group_type": best_ndcg_row["group_type"],
                "fairness_metric": "worst_member_satisfaction",
                "best_ndcg_method": best_ndcg_row["method"],
                "best_fairness_method": best_worst_row["method"],
                "best_ndcg_value": float(best_ndcg_row["group_ndcg_10"]),
                "fair_method_ndcg_value": float(best_worst_row["group_ndcg_10"]),
                "best_ndcg_fairness_value": float(best_ndcg_row["worst_member_satisfaction"]),
                "fair_method_fairness_value": float(best_worst_row["worst_member_satisfaction"]),
                "ndcg_difference": float(best_ndcg_row["group_ndcg_10"] - best_worst_row["group_ndcg_10"]),
                "fairness_difference": float(best_worst_row["worst_member_satisfaction"] - best_ndcg_row["worst_member_satisfaction"]),
            }
        )

    minority_part = group_part.dropna(subset=["minority_cluster_satisfaction"])

    if len(minority_part) > 0:
        best_minority_row = minority_part.loc[minority_part["minority_cluster_satisfaction"].idxmax()]

        if best_ndcg_row["method"] != best_minority_row["method"]:
            counterexample_rows.append(
                {
                    "group_id": group_id,
                    "group_type": best_ndcg_row["group_type"],
                    "fairness_metric": "minority_cluster_satisfaction",
                    "best_ndcg_method": best_ndcg_row["method"],
                    "best_fairness_method": best_minority_row["method"],
                    "best_ndcg_value": float(best_ndcg_row["group_ndcg_10"]),
                    "fair_method_ndcg_value": float(best_minority_row["group_ndcg_10"]),
                    "best_ndcg_fairness_value": float(best_ndcg_row["minority_cluster_satisfaction"]),
                    "fair_method_fairness_value": float(best_minority_row["minority_cluster_satisfaction"]),
                    "ndcg_difference": float(best_ndcg_row["group_ndcg_10"] - best_minority_row["group_ndcg_10"]),
                    "fairness_difference": float(best_minority_row["minority_cluster_satisfaction"] - best_ndcg_row["minority_cluster_satisfaction"]),
                }
            )

counterexamples_df = pd.DataFrame(counterexample_rows)

if len(counterexamples_df) > 0:
    counterexamples_df = counterexamples_df[
        (counterexamples_df["fairness_difference"] > 1e-12)
        & (counterexamples_df["ndcg_difference"] > 1e-12)
    ].copy()

    counterexamples_df = counterexamples_df.sort_values(
        by=["fairness_difference", "ndcg_difference"],
        ascending=[False, False],
    ).reset_index(drop=True)

display(counterexamples_df.head(10))
print("meaningful counterexamples found:", len(counterexamples_df))

,group_id,group_type,fairness_metric,best_ndcg_method,best_fairness_method,best_ndcg_value,fair_method_ndcg_value,best_ndcg_fairness_value,fair_method_fairness_value,ndcg_difference,fairness_difference
0,test_group_0135,diverse_balanced,jain_fairness,rating_user_knn_least_misery,mf_least_misery,0.180390,0.171339,0.333333,0.858286,0.009050,0.524952
1,test_group_0145,diverse_balanced,jain_fairness,mf_least_misery,mf_dbscan_minority_nash,0.274876,0.174371,0.333333,0.662154,0.100505,0.328821
2,test_group_0031,homogeneous_cluster_0,jain_fairness,mf_least_misery,rating_user_knn_average,0.371854,0.154574,0.333333,0.659849,0.217280,0.326516
3,test_group_0011,homogeneous_cluster_0,jain_fairness,mf_dbscan_minority_nash,hybrid_mf_bayes_least_misery,0.305235,0.298490,0.333333,0.634371,0.006745,0.301037
4,test_group_0049,homogeneous_cluster_0,jain_fairness,hybrid_mf_bayes_average,mf_least_misery,0.302404,0.278198,0.541195,0.635075,0.024206,0.093880
5,test_group_0135,diverse_balanced,worst_member_satisfaction,rating_user_knn_least_misery,mf_least_misery,0.180390,0.171339,0.000000,0.066254,0.009050,0.066254
6,test_group_0167,diverse_minority,jain_fairness,hybrid_mf_bayes_average,mf_nash_welfare,0.179477,0.135685,0.634190,0.666301,0.043792,0.032111
7,test_group_0083,homogeneous_cluster_1,jain_fairness,mf_average,mf_least_misery,0.173187,0.161043,0.576806,0.604249,0.012144,0.027443
8,test_group_0098,homogeneous_cluster_1,jain_fairness,hybrid_mf_bayes_average,hybrid_mf_bayes_least_misery,0.297369,0.239441,0.566938,0.592658,0.057929,0.025721
9,test_group_0047,homogeneous_cluster_0,jain_fairness,hybrid_mf_bayes_average,mf_average,0.183410,0.179477,0.531795,0.540662,0.003933,0.008868


meaningful counterexamples found: 10


### 9.4 Catalog coverage

In [25]:
coverage_rows = []

for key_values, rec_part in recommendations_df.groupby(["score_source", "aggregation", "method"]):
    score_source, aggregation, method = key_values
    unique_items = rec_part["item_idx"].nunique()

    coverage_rows.append(
        {
            "score_source": score_source,
            "aggregation": aggregation,
            "method": method,
            "unique_recommended_items": int(unique_items),
            "catalog_coverage": float(unique_items / n_items),
        }
    )

coverage_df = pd.DataFrame(coverage_rows).sort_values(
    by="catalog_coverage",
    ascending=False,
).reset_index(drop=True)

display(coverage_df)

,score_source,aggregation,method,unique_recommended_items,catalog_coverage
0,rating_user_knn,dbscan_minority_nash,rating_user_knn_dbscan_minority_nash,501,0.154154
1,rating_user_knn,average,rating_user_knn_average,463,0.142462
2,rating_user_knn,tuned_fairness,rating_user_knn_tuned_fairness,463,0.142462
3,rating_user_knn,nash_welfare,rating_user_knn_nash_welfare,460,0.141538
4,rating_user_knn,least_misery,rating_user_knn_least_misery,390,0.120000
5,mf,least_misery,mf_least_misery,338,0.104000
6,hybrid_mf_bayes,least_misery,hybrid_mf_bayes_least_misery,285,0.087692
7,mf,dbscan_minority_nash,mf_dbscan_minority_nash,274,0.084308
8,mf,nash_welfare,mf_nash_welfare,251,0.077231
9,hybrid_mf_bayes,dbscan_minority_nash,hybrid_mf_bayes_dbscan_minority_nash,246,0.075692


# Save Step 6 outputs

In [26]:
step4_pairwise_metrics.to_csv(step5_6_dir / "step6_pairwise_metrics_from_step4.csv", index=False)

group_metrics_df.to_csv(step5_6_dir / "step6_group_metrics.csv", index=False)
member_metrics_df.to_csv(step5_6_dir / "step6_member_metrics.csv", index=False)
individual_metrics_df.to_csv(step5_6_dir / "step6_individual_metrics.csv", index=False)
individual_summary_df.to_csv(step5_6_dir / "step6_individual_summary.csv", index=False)

method_summary_df.to_csv(step5_6_dir / "step6_method_summary.csv", index=False)
group_type_summary_df.to_csv(step5_6_dir / "step6_group_type_summary.csv", index=False)
minority_summary_df.to_csv(step5_6_dir / "step6_minority_summary.csv", index=False)
diversity_summary_df.to_csv(step5_6_dir / "step6_diversity_summary.csv", index=False)
tradeoff_summary_df.to_csv(step5_6_dir / "step6_tradeoff_summary.csv", index=False)
counterexamples_df.to_csv(step5_6_dir / "step6_counterexamples.csv", index=False)
coverage_df.to_csv(step5_6_dir / "step6_coverage.csv", index=False)

group_metrics_df.to_parquet(step5_6_dir / "step6_group_metrics.parquet", index=False)
member_metrics_df.to_parquet(step5_6_dir / "step6_member_metrics.parquet", index=False)
individual_metrics_df.to_parquet(step5_6_dir / "step6_individual_metrics.parquet", index=False)
individual_summary_df.to_parquet(step5_6_dir / "step6_individual_summary.parquet", index=False)

method_summary_df.to_parquet(step5_6_dir / "step6_method_summary.parquet", index=False)
group_type_summary_df.to_parquet(step5_6_dir / "step6_group_type_summary.parquet", index=False)
minority_summary_df.to_parquet(step5_6_dir / "step6_minority_summary.parquet", index=False)
diversity_summary_df.to_parquet(step5_6_dir / "step6_diversity_summary.parquet", index=False)
tradeoff_summary_df.to_parquet(step5_6_dir / "step6_tradeoff_summary.parquet", index=False)
counterexamples_df.to_parquet(step5_6_dir / "step6_counterexamples.parquet", index=False)
coverage_df.to_parquet(step5_6_dir / "step6_coverage.parquet", index=False)

print("saved all step 6 outputs to:", step5_6_dir)

saved all step 6 outputs to: results_step5_6


# Final Step 5–6 Summary: Group Recommendation, Fairness, Diversity, and Coverage

Steps 5 and 6 evaluate how the individual recommendation models behave in a group recommendation setting. Groups are constructed using the DBSCAN user clusters from Step 3, and recommendation methods are evaluated using relevance, fairness, minority satisfaction, diversity, and catalog coverage metrics.

## Step 5: Group construction

The final group setup uses:

| Parameter | Value |
|---|---:|
| Group size | 3 |
| Test groups per type | 50 |
| Total test groups | 200 |
| Total test group members | 600 |
| User reuse across test groups | no reuse |

Four group types are constructed:

1. homogeneous groups from DBSCAN cluster 0;
2. homogeneous groups from DBSCAN cluster 1;
3. diverse balanced groups combining cluster 0, cluster 1, and DBSCAN noise;
4. diverse minority groups with a 2:1 cluster composition.

This design allows the evaluation to compare easy homogeneous settings against harder diverse and minority-sensitive settings.

## Recommendation methods

The tested group aggregation methods include:

- average aggregation;
- least misery;
- tuned fairness aggregation;
- Nash welfare aggregation;
- DBSCAN minority-aware Nash aggregation.

The tested score sources are:

- MF;
- rating-user-kNN;
- hybrid MF + Bayesian.

## Validation tuning result

The validation tuning selected:

| Parameter | Selected value |
|---|---:|
| `lambda_std` for tuned fairness | 0.0 |
| `alpha_minority` for DBSCAN minority-aware Nash | 2.0 |

The selected `lambda_std = 0.0` is an important result. It means that, under the validation selection rule, adding the fairness penalty did not improve the fairness/relevance trade-off enough. Therefore, tuned fairness collapses to average aggregation in the final test results.

The selected `alpha_minority = 2.0` suggests that minority-aware reranking looked useful on validation, but this must be checked against test performance rather than assumed to generalize.

## Step 6: Individual recommendation results

| Score source | Users | Precision@10 | Recall@10 | Hit rate@10 | NDCG@10 |
|---|---:|---:|---:|---:|---:|
| MF | 4,623 | 0.0372 | 0.0346 | 0.2418 | 0.0434 |
| Rating-user-kNN | 4,623 | 0.0053 | 0.0044 | 0.0482 | 0.0062 |

MF clearly dominates kNN in individual top-10 recommendation quality. The absolute NDCG values are not high, but they are evaluated under a strict temporal held-out setting and a top-10 ranking task.

## Step 6: Main group recommendation results

| Method | Group NDCG@10 | Mean member NDCG@10 | Jain fairness | Zero satisfaction share | Member hit share |
|---|---:|---:|---:|---:|---:|
| MF + Nash welfare | 0.0677 | 0.0311 | 0.1583 | 0.8383 | 0.1617 |
| MF + average | 0.0663 | 0.0303 | 0.1549 | 0.8417 | 0.1583 |
| MF + tuned fairness | 0.0663 | 0.0303 | 0.1549 | 0.8417 | 0.1583 |
| MF + DBSCAN minority Nash | 0.0655 | 0.0304 | 0.1566 | 0.8400 | 0.1600 |
| Hybrid MF+Bayes + Nash welfare | 0.0620 | 0.0273 | 0.1418 | 0.8550 | 0.1450 |
| MF + least misery | 0.0554 | 0.0265 | 0.1240 | 0.8717 | 0.1283 |
| kNN + average | 0.0324 | 0.0142 | 0.1006 | 0.8983 | 0.1017 |

The best overall test method is **MF + Nash welfare**. It provides the strongest combination of group NDCG, member NDCG, Jain fairness, and member hit share.

However, the improvement over MF average aggregation is small:

- group NDCG improves by about `0.00135`;
- Jain fairness improves by about `0.00338`;
- member hit share improves by about `0.00333`;
- zero satisfaction share improves only from `0.8417` to `0.8383`.

Therefore, MF + Nash welfare is the best final method, but the gain should be described as modest rather than dramatic.

## Fairness and minority satisfaction caveat

The main limitation of the group recommendation results is the high zero-satisfaction rate. Even the best methods leave around 84% of member-level cases with zero satisfaction at top 10. This shows that group recommendation is substantially harder than individual recommendation in this setup.

Least misery does not solve this problem. It slightly targets the worst member in principle, but in the final results it reduces relevance and does not produce a strong fairness improvement.

The DBSCAN minority-aware method also does not consistently improve minority outcomes on the test set. In the diverse minority groups, MF average gives minority satisfaction of about `0.0351`, while MF DBSCAN minority Nash gives about `0.0303`. The minority zero-satisfaction rate also worsens from `0.82` to `0.84`.

This means the minority-aware method should be presented as an experimental reranking strategy, not as a confirmed fairness improvement.

## Coverage and diversity trade-off

The coverage results show an important trade-off:

| Method | Unique recommended items | Catalog coverage |
|---|---:|---:|
| kNN + DBSCAN minority Nash | 501 | 15.4% |
| kNN + average | 463 | 14.2% |
| MF + least misery | 338 | 10.4% |
| MF + DBSCAN minority Nash | 274 | 8.4% |
| MF + Nash welfare | 251 | 7.7% |
| MF + average | 224 | 6.9% |
| Hybrid MF+Bayes + average | 211 | 6.5% |

MF gives the strongest relevance, but kNN covers a much broader part of the catalog. This is one of the clearest trade-offs in the final evaluation: better ranking quality does not automatically imply better catalog coverage.

## Counterexamples

The counterexample analysis is important because it shows that the method with the best group utility is not always the method with the best fairness for every group. These cases support the broader conclusion that relevance and fairness objectives can conflict in group recommendation.

## Step 5–6 conclusion

The group recommendation pipeline is methodologically strong: groups are constructed from DBSCAN clusters, users are not reused across test groups, multiple aggregation rules are compared, and evaluation includes relevance, fairness, minority satisfaction, diversity, and coverage.

The best final method is **MF + Nash welfare aggregation**. However, the final report should interpret this result cautiously. Its improvement over average aggregation is small, zero-satisfaction rates remain high, and the DBSCAN minority-aware method does not consistently improve minority satisfaction on the test set.

The strongest conclusion is therefore not that the project fully solves group fairness, but that it demonstrates a complete and well-validated framework for measuring the relevance-fairness-coverage trade-offs in group recommendation.